# 09 — Exploração, Auditoria e Tratamento Embrapa BRLUC

## Projeto AgroESG — Soja | Centro-Oeste e Sul

Este notebook audita e prepara os dados do BRLUC
(Brazilian Land Use Change) utilizados no Projeto AgroESG.

Nesta etapa serão comparados:

1. os arquivos originais associados ao BRLUC 2.1;
2. os arquivos derivados produzidos pela equipe;
3. os indicadores relevantes para o recorte de soja nas regiões
   Centro-Oeste e Sul.

## Escopo do projeto

- Cultura: soja
- Regiões: Centro-Oeste e Sul
- UFs: DF, GO, MT, MS, PR, RS e SC
- Unidade espacial principal: município
- Fonte: Embrapa / BRLUC

## Objetivos

- documentar a origem dos dados;
- identificar o período de referência utilizado;
- compreender as unidades e os indicadores do BRLUC;
- auditar os arquivos derivados produzidos pela equipe;
- verificar possíveis inconsistências de escala, filtros e estrutura;
- avaliar os dados raster originais sem carregá-los integralmente em memória;
- produzir uma camada Curated adequada à integração posterior com
  as demais bases do projeto.

In [1]:
# ============================================================
# IMPORTAÇÕES E DIRETÓRIOS — BRLUC
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)


BRLUC_RAW_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "embrapa_brluc"
)


BRLUC_ORIGINAL_DIR = (
    BRLUC_RAW_DIR
    / "original_brluc_2_1"
)


print("Diretório BRLUC:")
print(BRLUC_RAW_DIR)

print("\nDiretório original BRLUC 2.1:")
print(BRLUC_ORIGINAL_DIR)

print("\nDiretórios encontrados:")
print("BRLUC:", BRLUC_RAW_DIR.exists())
print("Original:", BRLUC_ORIGINAL_DIR.exists())

Diretório BRLUC:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\embrapa_brluc

Diretório original BRLUC 2.1:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\embrapa_brluc\original_brluc_2_1

Diretórios encontrados:
BRLUC: True
Original: True


In [2]:
# ============================================================
# INVENTÁRIO DOS ARQUIVOS BRLUC
# ============================================================

print("=" * 70)
print("ARQUIVOS DERIVADOS — EQUIPE")
print("=" * 70)

for arquivo in sorted(BRLUC_RAW_DIR.glob("*.csv")):

    tamanho_mb = (
        arquivo.stat().st_size
        / (1024 * 1024)
    )

    print(
        f"{arquivo.name} -> {tamanho_mb:.2f} MB"
    )


print("\n" + "=" * 70)
print("ARQUIVOS ORIGINAIS — BRLUC 2.1")
print("=" * 70)

for arquivo in sorted(BRLUC_ORIGINAL_DIR.iterdir()):

    if arquivo.is_file():

        tamanho_mb = (
            arquivo.stat().st_size
            / (1024 * 1024)
        )

        print(
            f"{arquivo.name} -> {tamanho_mb:.2f} MB"
        )

ARQUIVOS DERIVADOS — EQUIPE
brluc_area_conversao_soja_municipal.csv -> 0.20 MB
brluc_area_conversao_soja_sul_centro_oeste.csv -> 0.09 MB
brluc_emissao_absoluta_soja_municipal.csv -> 0.26 MB
brluc_emissao_absoluta_soja_municipal_completo.csv -> 0.26 MB
brluc_estoque_carbono_municipal.csv -> 0.14 MB
brluc_estoque_carbono_sul_centro_oeste.csv -> 0.00 MB
brluc_percentual_conversao_soja_municipal.csv -> 0.20 MB
brluc_taxa_conversao_soja_municipal_completo.csv -> 0.26 MB
brluc_taxa_conversao_soja_sul_centro_oeste.csv -> 0.12 MB
brluc_taxa_emissao_soja_municipal_completo.csv -> 0.26 MB
brluc_taxa_emissao_soja_sul_centro_oeste.csv -> 0.10 MB

ARQUIVOS ORIGINAIS — BRLUC 2.1
Batistaetal_Appendix_A_REDAPE_v3.xlsx -> 16.65 MB
Batistaetal_Appendix_B_REDAPE_v2.xlsx -> 59.33 MB
Batistaetal_Appendix_C_REDAPE_v2.xlsx -> 55.79 MB
brCveg.tif -> 2523.03 MB
MANIFEST.TXT -> 0.00 MB
SOCref_trat.tif -> 0.25 MB


In [3]:
# ============================================================
# LEITURA DO MANIFEST — BRLUC 2.1
# ============================================================

ARQUIVO_MANIFEST = (
    BRLUC_ORIGINAL_DIR
    / "MANIFEST.txt"
)


print("Arquivo:")
print(ARQUIVO_MANIFEST)

print("\nExiste:")
print(ARQUIVO_MANIFEST.exists())


# Tentar encodings comuns
conteudo_manifest = None

for encoding in [
    "utf-8",
    "utf-8-sig",
    "latin-1",
    "cp1252"
]:

    try:

        conteudo_manifest = (
            ARQUIVO_MANIFEST
            .read_text(
                encoding=encoding
            )
        )

        print(
            f"\nEncoding utilizado: {encoding}"
        )

        break

    except UnicodeDecodeError:
        pass


print("\n" + "=" * 70)
print("CONTEÚDO DO MANIFEST")
print("=" * 70)

print(
    conteudo_manifest
)

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\embrapa_brluc\original_brluc_2_1\MANIFEST.txt

Existe:
True

Encoding utilizado: utf-8

CONTEÚDO DO MANIFEST
Batistaetal_Appendix_A_REDAPE_v3.xlsx (application/vnd.openxmlformats-officedocument.spreadsheetml.sheet) 17455369 bytes.
Batistaetal_Appendix_B_REDAPE_v2.xlsx (application/vnd.openxmlformats-officedocument.spreadsheetml.sheet) 62208783 bytes.
Batistaetal_Appendix_C_REDAPE_v2.xlsx (application/vnd.openxmlformats-officedocument.spreadsheetml.sheet) 58499532 bytes.
brCveg.tif (image/tiff) 2645588637 bytes.
SOCref_trat.tif (image/tiff) 264397 bytes.



In [4]:
# ============================================================
# INVENTÁRIO DAS PLANILHAS EXCEL — BRLUC 2.1
# ============================================================

arquivos_excel_brluc = sorted(
    BRLUC_ORIGINAL_DIR.glob("*.xlsx")
)


print(
    "Quantidade de arquivos Excel:"
)

print(
    len(
        arquivos_excel_brluc
    )
)


estrutura_excel_brluc = {}


for arquivo in arquivos_excel_brluc:

    print("\n" + "=" * 70)
    print(arquivo.name)
    print("=" * 70)

    excel = pd.ExcelFile(
        arquivo,
        engine="openpyxl"
    )

    estrutura_excel_brluc[
        arquivo.name
    ] = excel.sheet_names

    print(
        "Abas:"
    )

    for indice, aba in enumerate(
        excel.sheet_names,
        start=1
    ):

        print(
            f"{indice}. {aba}"
        )

Quantidade de arquivos Excel:
3

Batistaetal_Appendix_A_REDAPE_v3.xlsx
Abas:
1. Introduction
2. Legend(Subtitle)
3. Management Premises
4. 4CNdata
5. Uncertainty
6. CvegNative
7. SOCref
8. IPCCdata
9. ClimateZones
10. IBGE-PEVSdata
11. IBGE-CensoData
12. CONABdata
13. LAPIGdata
14. CvegPerennials
15. CvegCalc
16. SOCfactorsCalc
17. Table3Calc

Batistaetal_Appendix_B_REDAPE_v2.xlsx
Abas:
1. Introduction
2. Classes
3. C_total_Municipal_level
4. SOC_Municipal_ level
5. Cveg_Municipal_level
6. C_total_Microregional_level
7. SOC_Microregional_level
8. Cveg_Microregional_level
9. C_total_State_level
10. SOC_State_level
11. Cveg_State_level
12. C_total_National_level
13. SOC_National_level
14. Cveg_National_level
15. Unc_decomp_Municipal_level
16. Unc_decomp_Microregional_level
17. Unc_decomp_State_level
18. Unc_decomp_National_level

Batistaetal_Appendix_C_REDAPE_v2.xlsx
Abas:
1. Introduction
2. CO2_Municipal_level
3. CO2_Temporary_Municipal_level
4. CO2_Permanent_Municipal_level
5. CO2_Micr

In [5]:
# ============================================================
# AMOSTRA DAS ABAS — SEM CARREGAR OS EXCEL COMPLETOS
# ============================================================

for arquivo in arquivos_excel_brluc:

    print("\n" + "#" * 80)
    print(arquivo.name)
    print("#" * 80)

    excel = pd.ExcelFile(
        arquivo,
        engine="openpyxl"
    )

    for aba in excel.sheet_names:

        print("\n" + "-" * 70)
        print(f"ABA: {aba}")
        print("-" * 70)

        amostra = pd.read_excel(
            arquivo,
            sheet_name=aba,
            nrows=8,
            engine="openpyxl"
        )

        print(
            "Colunas:"
        )

        print(
            amostra.columns.tolist()
        )

        print(
            "\nDimensão da amostra:"
        )

        print(
            amostra.shape
        )

        display(
            amostra
        )


################################################################################
Batistaetal_Appendix_A_REDAPE_v3.xlsx
################################################################################

----------------------------------------------------------------------
ABA: Introduction
----------------------------------------------------------------------
Colunas:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18']

Dimensão da amostra:
(8, 19)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,NaN,Appendix A | Supplementary Data of Batista et ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,"Welcome to Appendix A, which contains the inpu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data generated are available for easier visual...
3,NaN,"BATISTA, A. M.; GOMES, L. E. S.; PAZIANOTTO, R...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brluc.cnpma.embrapa.br



----------------------------------------------------------------------
ABA: Legend(Subtitle)
----------------------------------------------------------------------
Colunas:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']

Dimensão da amostra:
(8, 15)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Table S1. Correspondence between legend and cl...,NaN,NaN,NaN,NaN,Table S2. Description of the 44 BRLUC C stock ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,MapBiomas Collection 9,BRLUC land-use class,BRLUC Cstock class,NaN,NaN,Major Land Use,Land Use Status,Crop Type or Species,Management Practice,Practice Detail or C Input Intensity,Class Name,Examples,NaN,Separador
4,NaN,1. Forest,Delete,NaN,NaN,NaN,Natural land,average,NaN,NaN,NaN,"Natural land, average","Weighted avg. of forest land, natural and gras...",NaN,NaN
5,NaN,1.1. Forest Formation,"Natural land, average","Natural land, average",NaN,NaN,Forest land,planted,eucalyptus,NaN,NaN,"Forest land, planted, eucalyptus",NaN,NaN,","
6,NaN,1.2. Savanna Formation,"Natural land, average","Natural land, average",NaN,NaN,Forest land,planted,pinus,NaN,NaN,"Forest land, planted, pinus",NaN,NaN,NaN
7,NaN,1.3. Mangrove,"Natural land, average","Natural land, average",NaN,NaN,Forest land,planted,other broadleaf,NaN,NaN,"Forest land, planted, other broadleaf",NaN,NaN,NaN



----------------------------------------------------------------------
ABA: Management Premises
----------------------------------------------------------------------
Colunas:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22']

Dimensão da amostra:
(8, 23)


...

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Category,Permanent crop,from Temporary,from Soybean,from Sugarcane,from Permanent,from Pasture,from Forestry,from Natural,Area1_t1,Absolute emissions,Absolute emissions_sd,Absolute emissions_ci95low,Absolute emissions_ci95upp,Area1e2_t1,Emission rates,Emission rates_sd,Emission rates_ci95low,Emission rates_ci95upp
3,NaN,,,,,,,,,,,CO2 abs,SE,CI95% Low,CI95% Upp,,CO2 rate,SE,CI95% Low,CI95% Upp
4,NaN,Permanent,Annatto,20110.545258,94.038693,174.611205,42257.837565,159229.699223,467.772394,17003.988701,239338.49304,-954742.881541,322542.712767,-1586926.598564,-322559.164517,239338.49304,-3.98909,1.347642,-6.63047,-1.347711
5,NaN,Permanent,Apple,55177.26459,138.867517,297.78149,33960.913991,53537.198654,14.614536,5598.225519,148724.866298,-582885.765683,143314.018323,-863781.241595,-301990.28977,148724.866298,-3.919222,0.963618,-5.807914,-2.03053
6,NaN,Permanent,Avocado,243684.113971,1955.272646,7169.085177,270405.522929,562676.56689,295.241746,62674.433499,1148860.236859,-4743047.080286,1295757.087234,-7282730.971264,-2203363.189308,1148860.236859,-4.12848,1.127863,-6.339092,-1.917869
7,NaN,Permanent,Banana,727265.615457,2775.917421,6958.047616,785045.304436,1759645.635971,925.084311,350790.72483,3633406.330042,-14033407.488407,4594416.787702,-23038464.392304,-5028350.584511,3633406.330042,-3.862328,1.264493,-6.340734,-1.383922



----------------------------------------------------------------------
ABA: Comparisons
----------------------------------------------------------------------
Colunas:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36']

Dimensão da amostra:
(8, 37)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36
0,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Comparison of DLUC-CO2 balance between scenari...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,BRLUC 2.1+MP (management practice changes were...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,CI95% Low,CI95% Upp,2.1 is higher?,2.0 is out of 95%,NaN,Summary,n,% 2.1 is higher,2.0 is out of 95%
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,BRLUC 2.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.491038,0.741719,2.240357,0,0,NaN,Crops,4,0.25,0
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-3.925244,-6.380593,-1.469896,0,0,NaN,Temp crops,31,0.096774,0.032258



----------------------------------------------------------------------
ABA: Land_Use_Classes
----------------------------------------------------------------------
Colunas:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5']

Dimensão da amostra:
(8, 6)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Land Use (BRLUC),Land Use Class,NaN,NaN,NaN
3,NaN,,Previous (at 2000; t0),NaN,NaN,Actual (at 2019; t1)
4,NaN,Temporary,"Cropland, temporary, generic, average tillage ...",NaN,NaN,"Cropland, temporary, generic, average tillage ..."
5,NaN,Soybean,"Cropland, temporary, generic, average tillage ...",NaN,NaN,"Cropland, temporary, generic, average tillage ..."
6,NaN,Sugarcane,"Cropland, temporary, sugarcane, average 2003",NaN,NaN,"Cropland, temporary, sugarcane, average 2023"
7,NaN,Permanent,"Cropland, permanent, generic, average tillage,...",NaN,NaN,"Cropland, permanent, generic, average tillage,..."


In [6]:
# ============================================================
# APPENDIX C — ABAS PRIORITÁRIAS PARA O PROJETO
# ============================================================

ARQUIVO_APPENDIX_C = (
    BRLUC_ORIGINAL_DIR
    / "Batistaetal_Appendix_C_REDAPE_v2.xlsx"
)


ABAS_C_PRIORITARIAS = [
    "CO2_Municipal_level",
    "CO2_Temporary_Municipal_level",
    "Land_Use_Classes"
]


for aba in ABAS_C_PRIORITARIAS:

    print("\n" + "=" * 80)
    print(aba)
    print("=" * 80)

    amostra = pd.read_excel(
        ARQUIVO_APPENDIX_C,
        sheet_name=aba,
        header=None,
        nrows=12,
        engine="openpyxl"
    )

    print(
        "Dimensão da amostra:",
        amostra.shape
    )

    display(
        amostra
    )


CO2_Municipal_level
Dimensão da amostra: (12, 25)


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Category,State code,State name,Microregion code,Microregion name (State abbreviation),Municipality code,Municipality name,from Temporary,from Soybean,...,Area1_t1,Absolute emissions,Absolute emissions_sd,Absolute emissions_ci95low,Absolute emissions_ci95upp,Area1e2_t1,Emission rates,Emission rates_sd,Emission rates_ci95low,Emission rates_ci95upp
4,NaN,,,,,,,,,,...,,CO2 abs,SE,CI95% Low,CI95% Upp,,CO2 rate,SE,CI95% Low,CI95% Upp
5,NaN,Temporary,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,3.144342,0,...,513.030065,5588.485775,1128.592659,3376.444164,7800.527387,522.183144,10.702157,2.161297,6.466015,14.938298
6,NaN,Temporary,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,249.844174,0,...,1362.822064,6598.200893,343.332933,5925.268345,7271.133441,1542.062142,4.278816,0.222645,3.842432,4.715201
7,NaN,Temporary,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,44.458459,0.34805,...,2910.480805,15302.199174,2959.750425,9501.08834,21103.310007,5144.757706,2.974328,0.575294,1.846751,4.101906
8,NaN,Temporary,11,Rondônia,11006,Cacoal (RO),1100049,Cacoal,132.026857,0,...,722.460004,1955.502585,149.150913,1663.166796,2247.838375,870.520646,2.24636,0.171335,1.910543,2.582177
9,NaN,Temporary,11,Rondônia,11008,Colorado do Oeste (RO),1100056,Cerejeiras,161.966502,25.431091,...,2718.66592,15341.594367,4540.612251,6441.994356,24241.194378,4930.401944,3.111632,0.920942,1.306586,4.916677



CO2_Temporary_Municipal_level
Dimensão da amostra: (12, 26)


,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Category,Temporary crop,State code,State name,Microregion code,Microregion name (State abbreviation),Municipality code,Municipality name,from Temporary,...,Area1_t1,Absolute emissions,Absolute emissions_sd,Absolute emissions_ci95low,Absolute emissions_ci95upp,Area1e2_t1,Emission rates,Emission rates_sd,Emission rates_ci95low,Emission rates_ci95upp
4,NaN,,,,,,,,,,...,,CO2 abs,SE,CI95% Low,CI95% Upp,,CO2 rate,SE,CI95% Low,CI95% Upp
5,NaN,Temporary,Barley,11,Rondônia,11001,Porto Velho (RO),1100205,Porto Velho,0,...,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN
6,NaN,Temporary,Barley,11,Rondônia,11001,Porto Velho (RO),1100338,Nova Mamoré,0,...,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN
7,NaN,Temporary,Barley,11,Rondônia,11001,Porto Velho (RO),1100452,Buritis,0,...,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN
8,NaN,Temporary,Barley,11,Rondônia,11001,Porto Velho (RO),1100700,Campo Novo de Rondônia,0,...,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN
9,NaN,Temporary,Barley,11,Rondônia,11001,Porto Velho (RO),1100809,Candeias do Jamari,0,...,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN



Land_Use_Classes
Dimensão da amostra: (12, 6)


,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Land Use (BRLUC),Land Use Class,NaN,NaN,NaN
4,NaN,,Previous (at 2000; t0),NaN,NaN,Actual (at 2019; t1)
5,NaN,Temporary,"Cropland, temporary, generic, average tillage ...",NaN,NaN,"Cropland, temporary, generic, average tillage ..."
6,NaN,Soybean,"Cropland, temporary, generic, average tillage ...",NaN,NaN,"Cropland, temporary, generic, average tillage ..."
7,NaN,Sugarcane,"Cropland, temporary, sugarcane, average 2003",NaN,NaN,"Cropland, temporary, sugarcane, average 2023"
8,NaN,Permanent,"Cropland, permanent, generic, average tillage,...",NaN,NaN,"Cropland, permanent, generic, average tillage,..."
9,NaN,Pasture,"Grassland, cultivated, average management 2003...",NaN,NaN,"Grassland, cultivated, average management 2023..."


In [7]:
# ============================================================
# APPENDIX C — CO2 MUNICIPAL COM CABEÇALHO ORIGINAL
# ============================================================

co2_municipal_original = pd.read_excel(
    ARQUIVO_APPENDIX_C,
    sheet_name="CO2_Municipal_level",
    header=[3, 4],
    engine="openpyxl"
)


print(
    "Dimensão:"
)

print(
    co2_municipal_original.shape
)


print(
    "\nQuantidade de colunas:"
)

print(
    co2_municipal_original.shape[1]
)


print(
    "\nEstrutura das colunas:"
)

for indice, coluna in enumerate(
    co2_municipal_original.columns
):

    print(
        indice,
        "->",
        coluna
    )

Dimensão:
(38983, 25)

Quantidade de colunas:
25

Estrutura das colunas:
0 -> ('Unnamed: 0_level_0', 'Unnamed: 0_level_1')
1 -> ('Category', ' ')
2 -> ('State code', ' ')
3 -> ('State name', ' ')
4 -> ('Microregion code', ' ')
5 -> ('Microregion name (State abbreviation)', ' ')
6 -> ('Municipality code', ' ')
7 -> ('Municipality name', ' ')
8 -> ('from Temporary', ' ')
9 -> ('from Soybean', ' ')
10 -> ('from Sugarcane', ' ')
11 -> ('from Permanent', ' ')
12 -> ('from Pasture', ' ')
13 -> ('from Forestry', ' ')
14 -> ('from Natural', ' ')
15 -> ('Area1_t1', ' ')
16 -> ('Absolute emissions', 'CO2 abs')
17 -> ('Absolute emissions_sd', 'SE')
18 -> ('Absolute emissions_ci95low', 'CI95% Low')
19 -> ('Absolute emissions_ci95upp', 'CI95% Upp')
20 -> ('Area1e2_t1', ' ')
21 -> ('Emission rates', 'CO2 rate')
22 -> ('Emission rates_sd', 'SE')
23 -> ('Emission rates_ci95low', 'CI95% Low')
24 -> ('Emission rates_ci95upp', 'CI95% Upp')


In [8]:
# ============================================================
# LOCALIZAÇÃO DA CATEGORIA SOYBEAN — APPENDIX C
# ============================================================

coluna_categoria = None


for coluna in co2_municipal_original.columns:

    textos = [
        str(nivel).strip().lower()
        for nivel in coluna
    ]

    if "category" in textos:

        coluna_categoria = coluna

        break


print(
    "Coluna identificada como Category:"
)

print(
    coluna_categoria
)


if coluna_categoria is not None:

    print(
        "\nCategorias encontradas:"
    )

    display(
        co2_municipal_original[
            coluna_categoria
        ]
        .value_counts(
            dropna=False
        )
    )


    soja_original = (
        co2_municipal_original[
            co2_municipal_original[
                coluna_categoria
            ]
            .astype("string")
            .str.strip()
            .eq(
                "Soybean"
            )
        ]
        .copy()
    )


    print(
        "\nRegistros Soybean:"
    )

    print(
        soja_original.shape
    )


    display(
        soja_original.head()
    )

Coluna identificada como Category:
('Category', ' ')

Categorias encontradas:


(Category,  )
Temporary    5569
Soybean      5569
Sugarcane    5569
Permanent    5569
Pasture      5569
Forestry     5569
Natural      5569
Name: count, dtype: int64


Registros Soybean:
(5569, 25)


,Unnamed: 0_level_0,Category,State code,State name,Microregion code,Microregion name (State abbreviation),Municipality code,Municipality name,from Temporary,from Soybean,...,Area1_t1,Absolute emissions,Absolute emissions_sd,Absolute emissions_ci95low,Absolute emissions_ci95upp,Area1e2_t1,Emission rates,Emission rates_sd,Emission rates_ci95low,Emission rates_ci95upp
,Unnamed: 0_level_1,,,,,,,,,,...,,CO2 abs,SE,CI95% Low,CI95% Upp,,CO2 rate,SE,CI95% Low,CI95% Upp
5569,NaN,Soybean,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,0.000000,0.000000,...,311.110790,1598.967390,265.270293,1079.037615,2118.897165,316.661383,5.049455,0.837710,3.407544,6.691366
5570,NaN,Soybean,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,0.000000,1.057436,...,3744.757576,22954.820124,1177.657829,20646.610779,25263.029468,4237.272820,5.417357,0.277928,4.872618,5.962097
5571,NaN,Soybean,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,157.752375,174.310774,...,21522.521655,104025.096710,19927.518276,64967.160888,143083.032531,38044.627865,2.734291,0.523793,1.707657,3.760926
5572,NaN,Soybean,11,Rondônia,11006,Cacoal (RO),1100049,Cacoal,2.016285,0.613716,...,740.580796,4666.784060,231.393511,4213.252778,5120.315341,892.355105,5.229739,0.259307,4.721498,5.737980
5573,NaN,Soybean,11,Rondônia,11008,Colorado do Oeste (RO),1100056,Cerejeiras,401.871093,1009.454416,...,31880.458018,89372.556326,24707.885264,40945.101210,137800.011443,57816.398482,1.545799,0.427351,0.708192,2.383407


In [9]:
# ============================================================
# LIMPEZA DA TABELA OFICIAL — SOYBEAN
# ============================================================

co2_municipal_limpo = (
    co2_municipal_original
    .copy()
)


# ------------------------------------------------------------
# Utilizar o primeiro nível do cabeçalho
# ------------------------------------------------------------

co2_municipal_limpo.columns = [
    coluna[0]
    if isinstance(coluna, tuple)
    else coluna
    for coluna in co2_municipal_limpo.columns
]


# ------------------------------------------------------------
# Remover a primeira coluna totalmente vazia
# ------------------------------------------------------------

colunas_totalmente_vazias = (
    co2_municipal_limpo
    .columns[
        co2_municipal_limpo
        .isna()
        .all()
    ]
    .tolist()
)


print(
    "Colunas totalmente vazias:"
)

print(
    colunas_totalmente_vazias
)


co2_municipal_limpo = (
    co2_municipal_limpo
    .drop(
        columns=colunas_totalmente_vazias
    )
)


print(
    "\nDimensão:"
)

print(
    co2_municipal_limpo.shape
)


print(
    "\nColunas finais:"
)

for coluna in co2_municipal_limpo.columns:

    print(
        coluna
    )

Colunas totalmente vazias:
['Unnamed: 0_level_0']

Dimensão:
(38983, 24)

Colunas finais:
Category
State code
State name
Microregion code
Microregion name (State abbreviation)
Municipality code
Municipality name
from Temporary
from Soybean
from Sugarcane
from Permanent
from Pasture
from Forestry
from Natural
Area1_t1
Absolute emissions
Absolute emissions_sd
Absolute emissions_ci95low
Absolute emissions_ci95upp
Area1e2_t1
Emission rates
Emission rates_sd
Emission rates_ci95low
Emission rates_ci95upp


In [10]:
# ============================================================
# RECORTE OFICIAL — SOJA | CENTRO-OESTE + SUL
# ============================================================

UFS_PROJETO = [
    "DF",
    "GO",
    "MS",
    "MT",
    "PR",
    "RS",
    "SC"
]


brluc_soja_original = (
    co2_municipal_limpo[
        co2_municipal_limpo[
            "Category"
        ]
        .astype("string")
        .str.strip()
        .eq(
            "Soybean"
        )
    ]
    .copy()
)


brluc_soja_original[
    "Municipality code"
] = (
    brluc_soja_original[
        "Municipality code"
    ]
    .astype("Int64")
    .astype("string")
    .str.zfill(7)
)


# UF a partir dos dois primeiros dígitos
MAPA_CODIGO_UF = {
    "53": "DF",
    "52": "GO",
    "50": "MS",
    "51": "MT",
    "41": "PR",
    "43": "RS",
    "42": "SC"
}


brluc_soja_original[
    "uf"
] = (
    brluc_soja_original[
        "Municipality code"
    ]
    .str[:2]
    .map(
        MAPA_CODIGO_UF
    )
)


brluc_soja_projeto = (
    brluc_soja_original[
        brluc_soja_original[
            "uf"
        ]
        .isin(
            UFS_PROJETO
        )
    ]
    .copy()
)


print(
    "Soja — Brasil:"
)

print(
    brluc_soja_original.shape
)


print(
    "\nSoja — Centro-Oeste + Sul:"
)

print(
    brluc_soja_projeto.shape
)


print(
    "\nMunicípios distintos:"
)

print(
    brluc_soja_projeto[
        "Municipality code"
    ]
    .nunique()
)


print(
    "\nDistribuição por UF:"
)

display(
    brluc_soja_projeto[
        "uf"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nDuplicatas de código IBGE:"
)

print(
    brluc_soja_projeto
    .duplicated(
        subset=[
            "Municipality code"
        ]
    )
    .sum()
)

Soja — Brasil:
(5569, 25)

Soja — Centro-Oeste + Sul:
(1658, 25)

Municípios distintos:
1658

Distribuição por UF:


uf
DF      1
GO    246
MS     79
MT    141
PR    399
RS    497
SC    295
Name: count, dtype: int64


Duplicatas de código IBGE:
0


In [11]:
# ============================================================
# INDICADORES OFICIAIS — BRLUC SOJA
# ============================================================

COLUNAS_BRLUC_PRIORITARIAS = [
    "Municipality code",
    "Municipality name",
    "State name",
    "uf",

    "from Temporary",
    "from Soybean",
    "from Sugarcane",
    "from Permanent",
    "from Pasture",
    "from Forestry",
    "from Natural",

    "Area1_t1",

    "Absolute emissions",
    "Absolute emissions_sd",
    "Absolute emissions_ci95low",
    "Absolute emissions_ci95upp",

    "Area1e2_t1",

    "Emission rates",
    "Emission rates_sd",
    "Emission rates_ci95low",
    "Emission rates_ci95upp"
]


brluc_soja_oficial = (
    brluc_soja_projeto[
        COLUNAS_BRLUC_PRIORITARIAS
    ]
    .copy()
)


print(
    "Dimensão:"
)

print(
    brluc_soja_oficial.shape
)


print(
    "\nTipos:"
)

display(
    brluc_soja_oficial.dtypes
)


print(
    "\nPrimeiros registros:"
)

display(
    brluc_soja_oficial.head()
)

Dimensão:
(1658, 21)

Tipos:


Municipality code             string[python]
Municipality name                     object
State name                            object
uf                                    object
from Temporary                       float64
from Soybean                         float64
from Sugarcane                       float64
from Permanent                       float64
from Pasture                         float64
from Forestry                        float64
from Natural                         float64
Area1_t1                             float64
Absolute emissions                   float64
Absolute emissions_sd                float64
Absolute emissions_ci95low           float64
Absolute emissions_ci95upp           float64
Area1e2_t1                           float64
Emission rates                       float64
Emission rates_sd                    float64
Emission rates_ci95low               float64
Emission rates_ci95upp               float64
dtype: object


Primeiros registros:


,Municipality code,Municipality name,State name,uf,from Temporary,from Soybean,from Sugarcane,from Permanent,from Pasture,from Forestry,...,Area1_t1,Absolute emissions,Absolute emissions_sd,Absolute emissions_ci95low,Absolute emissions_ci95upp,Area1e2_t1,Emission rates,Emission rates_sd,Emission rates_ci95low,Emission rates_ci95upp
9480,4100103,Abatiá,Paraná,PR,2488.168493,371.845266,349.938722,178.313661,1143.105958,0.658043,...,4694.811800,5012.993234,567.814905,3900.076021,6125.910447,9309.007371,0.538510,0.060996,0.418957,0.658063
9481,4100202,Adrianópolis,Paraná,PR,1.450002,1.302738,0.000000,0.256859,14.489060,0.569696,...,37.508712,663.440471,76.002699,514.475181,812.405761,42.796434,15.502237,1.775912,12.021450,18.983024
9482,4100301,Agudos do Sul,Paraná,PR,326.200765,1.287731,0.000000,0.118580,13.878543,0.000000,...,368.621819,788.841844,48.443502,693.892581,883.791107,376.743627,2.093843,0.128585,1.841816,2.345869
9483,4100400,Almirante Tamandaré,Paraná,PR,508.821336,17.580535,0.000000,1.823916,24.972276,0.000000,...,666.825185,3002.382074,357.596834,2301.492278,3703.271869,677.926603,4.428772,0.527486,3.394899,5.462644
9484,4100459,Altamira do Paraná,Paraná,PR,138.539817,0.000000,0.000000,0.146796,586.002505,0.000000,...,764.064713,1865.148758,215.346064,1443.070473,2287.227043,970.680374,1.921486,0.221851,1.486659,2.356313


In [12]:
# ============================================================
# AUDITORIA NUMÉRICA — BRLUC OFICIAL
# ============================================================

COLUNAS_NUMERICAS_BRLUC = [
    "from Temporary",
    "from Soybean",
    "from Sugarcane",
    "from Permanent",
    "from Pasture",
    "from Forestry",
    "from Natural",
    "Area1_t1",
    "Absolute emissions",
    "Absolute emissions_sd",
    "Absolute emissions_ci95low",
    "Absolute emissions_ci95upp",
    "Area1e2_t1",
    "Emission rates",
    "Emission rates_sd",
    "Emission rates_ci95low",
    "Emission rates_ci95upp"
]


resumo_numerico_brluc = (
    brluc_soja_oficial[
        COLUNAS_NUMERICAS_BRLUC
    ]
    .describe()
    .T
)


display(
    resumo_numerico_brluc
)


print(
    "\nValores negativos:"
)

for coluna in [
    "Absolute emissions",
    "Emission rates"
]:

    print(
        coluna,
        "->",
        (
            brluc_soja_oficial[
                coluna
            ] < 0
        ).sum()
    )


print(
    "\nValores nulos:"
)

display(
    brluc_soja_oficial[
        COLUNAS_NUMERICAS_BRLUC
    ]
    .isna()
    .sum()
)

,count,mean,std,min,25%,50%,75%,max
from Temporary,1658.0,3955.485197,9248.950487,0.000000,31.094916,827.028140,4049.524894,1.254075e+05
from Soybean,1658.0,5371.782042,18602.321787,0.000000,1.978428,153.630785,3437.902617,3.599788e+05
from Sugarcane,1658.0,25.409623,170.028812,0.000000,0.000000,0.000000,0.000000,2.443901e+03
from Permanent,1658.0,18.566917,70.187080,0.000000,0.000000,0.321020,8.565705,1.515560e+03
from Pasture,1658.0,4576.687309,14790.506477,0.000000,15.148668,319.419477,2224.637703,1.793273e+05
from Forestry,1658.0,27.742088,202.155455,0.000000,0.000000,0.000000,0.158866,6.364020e+03
from Natural,1658.0,2771.215727,13025.753772,0.000000,1.995599,42.217021,435.269340,2.399102e+05
Area1_t1,1658.0,16746.888903,42489.695547,0.000000,357.398811,3639.294134,14121.975354,5.834695e+05
Absolute emissions,1658.0,35309.477570,187109.044399,-3370.005124,249.884416,2115.756331,10411.859409,3.463402e+06
Absolute emissions_sd,1556.0,10140.369684,46871.430738,0.006081,145.559072,755.609256,3048.736723,8.054412e+05



Valores negativos:
Absolute emissions -> 17
Emission rates -> 17

Valores nulos:


from Temporary                  0
from Soybean                    0
from Sugarcane                  0
from Permanent                  0
from Pasture                    0
from Forestry                   0
from Natural                    0
Area1_t1                        0
Absolute emissions              0
Absolute emissions_sd         102
Absolute emissions_ci95low    102
Absolute emissions_ci95upp    102
Area1e2_t1                      0
Emission rates                  0
Emission rates_sd             118
Emission rates_ci95low        118
Emission rates_ci95upp        118
dtype: int64

In [13]:
# ============================================================
# CARREGAMENTO DOS CSVs DERIVADOS PRIORITÁRIOS
# ============================================================

ARQUIVO_TAXA_EMISSAO_EQUIPE = (
    BRLUC_RAW_DIR
    / "brluc_taxa_emissao_soja_sul_centro_oeste.csv"
)


ARQUIVO_AREA_CONVERSAO_EQUIPE = (
    BRLUC_RAW_DIR
    / "brluc_area_conversao_soja_sul_centro_oeste.csv"
)


ARQUIVO_EMISSAO_ABSOLUTA_EQUIPE = (
    BRLUC_RAW_DIR
    / "brluc_emissao_absoluta_soja_municipal.csv"
)


ARQUIVO_PERCENTUAL_CONVERSAO_EQUIPE = (
    BRLUC_RAW_DIR
    / "brluc_percentual_conversao_soja_municipal.csv"
)


arquivos_derivados_prioritarios = {
    "Taxa de emissão":
        ARQUIVO_TAXA_EMISSAO_EQUIPE,

    "Área de conversão":
        ARQUIVO_AREA_CONVERSAO_EQUIPE,

    "Emissão absoluta":
        ARQUIVO_EMISSAO_ABSOLUTA_EQUIPE,

    "Percentual de conversão":
        ARQUIVO_PERCENTUAL_CONVERSAO_EQUIPE
}


bases_derivadas_brluc = {}


for nome, arquivo in arquivos_derivados_prioritarios.items():

    base = pd.read_csv(
        arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        low_memory=False
    )

    bases_derivadas_brluc[
        nome
    ] = base

    print("\n" + "=" * 80)
    print(nome)
    print("=" * 80)

    print(
        "Arquivo:",
        arquivo.name
    )

    print(
        "Dimensão:",
        base.shape
    )

    print(
        "Colunas:"
    )

    print(
        base.columns.tolist()
    )

    display(
        base.head(3)
    )


Taxa de emissão
Arquivo: brluc_taxa_emissao_soja_sul_centro_oeste.csv
Dimensão: (1658, 9)
Colunas:
['Codigo_Municipio', 'Municipio', 'Estado', 'UF', 'Regiao', 'Cultura', 'Taxa_Emissao', 'IC_Inferior', 'IC_Superior']


,Codigo_Municipio,Municipio,Estado,UF,Regiao,Cultura,Taxa_Emissao,IC_Inferior,IC_Superior
0,5300108,Brasília,Distrito Federal,DF,Centro-Oeste,Soja,0.34,0.15,0.53
1,5200050,Abadia de Goiás,Goiás,GO,Centro-Oeste,Soja,0.13,0.06,0.20
2,5200100,Abadiânia,Goiás,GO,Centro-Oeste,Soja,1.03,0.72,1.34



Área de conversão
Arquivo: brluc_area_conversao_soja_sul_centro_oeste.csv
Dimensão: (1658, 7)
Colunas:
['codigo_municipio', 'municipio', 'estado', 'uf', 'regiao', 'cultura', 'area_conversao_ha']


,codigo_municipio,municipio,estado,uf,regiao,cultura,area_conversao_ha
0,5300108,Brasília,Distrito Federal,DF,Centro-Oeste,Soja,45600.42
1,5200050,Abadia de Goiás,Goiás,GO,Centro-Oeste,Soja,11.48
2,5200100,Abadiânia,Goiás,GO,Centro-Oeste,Soja,1439.25



Emissão absoluta
Arquivo: brluc_emissao_absoluta_soja_municipal.csv
Dimensão: (5570, 6)
Colunas:
['Codigo_Municipio', 'Municipio', 'Estado', 'Emissao_Absoluta', 'IC_Inferior', 'IC_Superior']


,Codigo_Municipio,Municipio,Estado,Emissao_Absoluta,IC_Inferior,IC_Superior
0,1100015,Alta Floresta D'Oeste,Rondônia,1598.97,1079.04,2118.90
1,1100379,Alto Alegre dos Parecis,Rondônia,3418.96,2290.00,4547.92
2,1100403,Alto Paraíso,Rondônia,62105.75,56436.17,67775.33



Percentual de conversão
Arquivo: brluc_percentual_conversao_soja_municipal.csv
Dimensão: (5570, 4)
Colunas:
['Codigo_Municipio', 'Municipio', 'Estado', 'Percentual_Conversao']


,Codigo_Municipio,Municipio,Estado,Percentual_Conversao
0,1100015,Alta Floresta D'Oeste,Rondônia,0.00
1,1100379,Alto Alegre dos Parecis,Rondônia,0.63
2,1100403,Alto Paraíso,Rondônia,0.00


In [14]:
# ============================================================
# AUDITORIA TERRITORIAL DOS CSVs DERIVADOS
# ============================================================

for nome, base in bases_derivadas_brluc.items():

    print("\n" + "=" * 80)
    print(nome)
    print("=" * 80)

    colunas_codigo = [
        coluna
        for coluna in base.columns
        if (
            "codigo" in coluna.lower()
            or
            "code" in coluna.lower()
            or
            "ibge" in coluna.lower()
        )
    ]

    print(
        "Possíveis colunas de código:"
    )

    print(
        colunas_codigo
    )

    print(
        "\nDimensão:"
    )

    print(
        base.shape
    )


Taxa de emissão
Possíveis colunas de código:
['Codigo_Municipio']

Dimensão:
(1658, 9)

Área de conversão
Possíveis colunas de código:
['codigo_municipio']

Dimensão:
(1658, 7)

Emissão absoluta
Possíveis colunas de código:
['Codigo_Municipio']

Dimensão:
(5570, 6)

Percentual de conversão
Possíveis colunas de código:
['Codigo_Municipio']

Dimensão:
(5570, 4)


In [15]:
# ============================================================
# REFERÊNCIA OFICIAL BRLUC — UNIVERSO MUNICIPAL DO PROJETO
# ============================================================

codigos_brluc_oficial = (
    brluc_soja_oficial[
        [
            "Municipality code",
            "Municipality name",
            "uf"
        ]
    ]
    .rename(
        columns={
            "Municipality code":
                "codigo_ibge",

            "Municipality name":
                "municipio"
        }
    )
    .copy()
)


codigos_brluc_oficial[
    "codigo_ibge"
] = (
    codigos_brluc_oficial[
        "codigo_ibge"
    ]
    .astype("string")
    .str.strip()
    .str.zfill(7)
)


print(
    "Municípios oficiais BRLUC no projeto:"
)

print(
    codigos_brluc_oficial[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    codigos_brluc_oficial
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


display(
    codigos_brluc_oficial.head()
)

Municípios oficiais BRLUC no projeto:
1658

Duplicatas:
0


,codigo_ibge,municipio,uf
9480,4100103,Abatiá,PR
9481,4100202,Adrianópolis,PR
9482,4100301,Agudos do Sul,PR
9483,4100400,Almirante Tamandaré,PR
9484,4100459,Altamira do Paraná,PR


In [16]:
# ============================================================
# DOCUMENTAÇÃO OFICIAL — APPENDIX C
# ============================================================

introducao_appendix_c = pd.read_excel(
    ARQUIVO_APPENDIX_C,
    sheet_name="Introduction",
    header=None,
    engine="openpyxl"
)


print(
    "Dimensão:"
)

print(
    introducao_appendix_c.shape
)


display(
    introducao_appendix_c
)

Dimensão:
(90, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Appendix C | Supplementary Data of Batista et ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,"Welcome to Appendix C, which contains the outp...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data are also available for easier visualizati...
4,NaN,"BATISTA, A. M.; GOMES, L. E. S.; PAZIANOTTO, R...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# ============================================================
# DOCUMENTAÇÃO — CLASSES DE USO DA TERRA
# ============================================================

land_use_classes = pd.read_excel(
    ARQUIVO_APPENDIX_C,
    sheet_name="Land_Use_Classes",
    header=None,
    engine="openpyxl"
)


print(
    "Dimensão:"
)

print(
    land_use_classes.shape
)


display(
    land_use_classes
)

Dimensão:
(26, 9)


,0,1,2,3,4,5,6,7,8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Land Use (BRLUC),Land Use Class,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,,Previous (at 2000; t0),NaN,NaN,Actual (at 2019; t1),NaN,NaN,NaN
5,NaN,Temporary,"Cropland, temporary, generic, average tillage ...",NaN,NaN,"Cropland, temporary, generic, average tillage ...",NaN,NaN,NaN
6,NaN,Soybean,"Cropland, temporary, generic, average tillage ...",NaN,NaN,"Cropland, temporary, generic, average tillage ...",NaN,NaN,NaN
7,NaN,Sugarcane,"Cropland, temporary, sugarcane, average 2003",NaN,NaN,"Cropland, temporary, sugarcane, average 2023",NaN,NaN,NaN
8,NaN,Permanent,"Cropland, permanent, generic, average tillage,...",NaN,NaN,"Cropland, permanent, generic, average tillage,...",NaN,NaN,NaN
9,NaN,Pasture,"Grassland, cultivated, average management 2003...",NaN,NaN,"Grassland, cultivated, average management 2023...",NaN,NaN,NaN


In [18]:
# ============================================================
# TABELA ESPECÍFICA DE CULTURAS TEMPORÁRIAS
# ============================================================

co2_temporary_original = pd.read_excel(
    ARQUIVO_APPENDIX_C,
    sheet_name="CO2_Temporary_Municipal_level",
    header=[3, 4],
    engine="openpyxl"
)


co2_temporary_original.columns = [
    coluna[0]
    if isinstance(coluna, tuple)
    else coluna
    for coluna in co2_temporary_original.columns
]


# Remover colunas totalmente vazias
colunas_vazias_temp = (
    co2_temporary_original
    .columns[
        co2_temporary_original
        .isna()
        .all()
    ]
    .tolist()
)


co2_temporary_original = (
    co2_temporary_original
    .drop(
        columns=colunas_vazias_temp
    )
)


print(
    "Dimensão:"
)

print(
    co2_temporary_original.shape
)


print(
    "\nColunas:"
)

print(
    co2_temporary_original.columns.tolist()
)


print(
    "\nCulturas temporárias:"
)

display(
    co2_temporary_original[
        "Temporary crop"
    ]
    .value_counts(
        dropna=False
    )
)

Dimensão:
(161501, 25)

Colunas:
['Category', 'Temporary crop', 'State code', 'State name', 'Microregion code', 'Microregion name (State abbreviation)', 'Municipality code', 'Municipality name', 'from Temporary', 'from Soybean', 'from Sugarcane', 'from Permanent', 'from Pasture', 'from Forestry', 'from Natural', 'Area1_t1', 'Absolute emissions', 'Absolute emissions_sd', 'Absolute emissions_ci95low', 'Absolute emissions_ci95upp', 'Area1e2_t1', 'Emission rates', 'Emission rates_sd', 'Emission rates_ci95low', 'Emission rates_ci95upp']

Culturas temporárias:


Temporary crop
Barley                 5569
Peanut                 5569
Watermelon             5569
Triticale              5569
Tomato                 5569
Tobacco                5569
Sweet potato           5569
Sunflower              5569
Sorghum                5569
Rye                    5569
Rice                   5569
Ramie                  5569
Potato                 5569
Pineapple              5569
Pea                    5569
Bean                   5569
Onion                  5569
Oat                    5569
Melon                  5569
Manioc                 5569
Malva                  5569
Maize                  5569
Jute                   5569
Garlic                 5569
Flax                   5569
Cotton (herbaceous)    5569
Castor bean            5569
Broad bean             5569
Wheat                  5569
Name: count, dtype: int64

In [19]:
# ============================================================
# SOYBEAN — TABELA DE CULTURAS TEMPORÁRIAS
# ============================================================

soja_temporary_original = (
    co2_temporary_original[
        co2_temporary_original[
            "Temporary crop"
        ]
        .astype("string")
        .str.strip()
        .eq(
            "Soybean"
        )
    ]
    .copy()
)


print(
    "Registros Soybean:"
)

print(
    soja_temporary_original.shape
)


print(
    "\nPrimeiros registros:"
)

display(
    soja_temporary_original.head()
)

Registros Soybean:
(0, 25)

Primeiros registros:


,Category,Temporary crop,State code,State name,Microregion code,Microregion name (State abbreviation),Municipality code,Municipality name,from Temporary,from Soybean,...,Area1_t1,Absolute emissions,Absolute emissions_sd,Absolute emissions_ci95low,Absolute emissions_ci95upp,Area1e2_t1,Emission rates,Emission rates_sd,Emission rates_ci95low,Emission rates_ci95upp


### Observação

A tabela `CO2_Temporary_Municipal_level` não contém registros para
`Soybean`.

A soja é tratada como categoria específica na aba
`CO2_Municipal_level`, portanto esta passa a ser a fonte oficial
utilizada para o recorte de soja neste notebook.

In [20]:
# ============================================================
# AUDITORIA DAS RELAÇÕES MATEMÁTICAS — APPENDIX C
# ============================================================

COLUNAS_ORIGEM_USO = [
    "from Temporary",
    "from Soybean",
    "from Sugarcane",
    "from Permanent",
    "from Pasture",
    "from Forestry",
    "from Natural"
]


brluc_soja_teste = (
    brluc_soja_oficial
    .copy()
)


# ------------------------------------------------------------
# Soma das áreas de origem
# ------------------------------------------------------------

brluc_soja_teste[
    "soma_areas_origem"
] = (
    brluc_soja_teste[
        COLUNAS_ORIGEM_USO
    ]
    .sum(axis=1)
)


brluc_soja_teste[
    "dif_soma_origem_area1"
] = (
    brluc_soja_teste[
        "soma_areas_origem"
    ]
    -
    brluc_soja_teste[
        "Area1_t1"
    ]
).abs()


# ------------------------------------------------------------
# Teste da fórmula da taxa de emissão
# ------------------------------------------------------------

brluc_soja_teste[
    "taxa_emissao_calculada"
] = (
    brluc_soja_teste[
        "Absolute emissions"
    ]
    /
    brluc_soja_teste[
        "Area1e2_t1"
    ]
    .replace(
        0,
        np.nan
    )
)


brluc_soja_teste[
    "dif_taxa_emissao"
] = (
    brluc_soja_teste[
        "taxa_emissao_calculada"
    ]
    -
    brluc_soja_teste[
        "Emission rates"
    ]
).abs()


print("=" * 70)
print("RELAÇÕES MATEMÁTICAS — BRLUC")
print("=" * 70)


print(
    "\nMaior diferença:"
    "\nSoma das áreas de origem × Area1_t1"
)

print(
    brluc_soja_teste[
        "dif_soma_origem_area1"
    ].max()
)


print(
    "\nRegistros com diferença > 0.000001:"
)

print(
    (
        brluc_soja_teste[
            "dif_soma_origem_area1"
        ]
        > 0.000001
    )
    .sum()
)


print(
    "\nMaior diferença:"
    "\nAbsolute emissions / Area1e2_t1 × Emission rates"
)

print(
    brluc_soja_teste[
        "dif_taxa_emissao"
    ].max()
)


print(
    "\nRegistros com diferença > 0.000001:"
)

print(
    (
        brluc_soja_teste[
            "dif_taxa_emissao"
        ]
        > 0.000001
    )
    .sum()
)

RELAÇÕES MATEMÁTICAS — BRLUC

Maior diferença:
Soma das áreas de origem × Area1_t1
9.313225746154785e-10

Registros com diferença > 0.000001:
0

Maior diferença:
Absolute emissions / Area1e2_t1 × Emission rates
9.592326932761353e-14

Registros com diferença > 0.000001:
0


In [21]:
# ============================================================
# HIPÓTESE — ÁREA DE CONVERSÃO PARA SOJA
# ============================================================

COLUNAS_ORIGEM_EXCETO_SOJA = [
    "from Temporary",
    "from Sugarcane",
    "from Permanent",
    "from Pasture",
    "from Forestry",
    "from Natural"
]


brluc_soja_teste[
    "area_conversao_candidata_ha"
] = (
    brluc_soja_teste[
        COLUNAS_ORIGEM_EXCETO_SOJA
    ]
    .sum(axis=1)
)


brluc_soja_teste[
    "area1_menos_soja_ha"
] = (
    brluc_soja_teste[
        "Area1_t1"
    ]
    -
    brluc_soja_teste[
        "from Soybean"
    ]
)


brluc_soja_teste[
    "dif_area_conversao_candidata"
] = (
    brluc_soja_teste[
        "area_conversao_candidata_ha"
    ]
    -
    brluc_soja_teste[
        "area1_menos_soja_ha"
    ]
).abs()


print(
    "Maior diferença entre:"
    "\n"
    "soma das origens diferentes de soja"
    "\n"
    "e"
    "\n"
    "Area1_t1 - from Soybean"
)

print(
    brluc_soja_teste[
        "dif_area_conversao_candidata"
    ].max()
)


print(
    "\nRegistros fora da tolerância:"
)

print(
    (
        brluc_soja_teste[
            "dif_area_conversao_candidata"
        ]
        > 0.000001
    )
    .sum()
)


display(
    brluc_soja_teste[
        [
            "Municipality code",
            "Municipality name",
            "uf",
            "Area1_t1",
            "from Soybean",
            "area_conversao_candidata_ha"
        ]
    ]
    .head(10)
)

Maior diferença entre:
soma das origens diferentes de soja
e
Area1_t1 - from Soybean
9.022187441587448e-10

Registros fora da tolerância:
0


,Municipality code,Municipality name,uf,Area1_t1,from Soybean,area_conversao_candidata_ha
9480,4100103,Abatiá,PR,4694.811800,371.845266,4322.966534
9481,4100202,Adrianópolis,PR,37.508712,1.302738,36.205975
9482,4100301,Agudos do Sul,PR,368.621819,1.287731,367.334087
9483,4100400,Almirante Tamandaré,PR,666.825185,17.580535,649.244650
9484,4100459,Altamira do Paraná,PR,764.064713,0.000000,764.064713
9485,4100509,Altônia,PR,5468.098215,141.222467,5326.875748
9486,4100608,Alto Paraná,PR,788.676790,4.863484,783.813306
9487,4100707,Alto Piquiri,PR,28905.541088,4071.245017,24834.296071
9488,4100806,Alvorada do Sul,PR,19008.272096,9311.821595,9696.450501
9489,4100905,Amaporã,PR,2278.756662,3.213496,2275.543166


In [22]:
# ============================================================
# AUDITORIA — ÁREA DE CONVERSÃO ORIGINAL × CSV DA EQUIPE
# ============================================================

area_equipe = (
    bases_derivadas_brluc[
        "Área de conversão"
    ]
    .copy()
)


area_equipe[
    "codigo_ibge"
] = (
    area_equipe[
        "codigo_municipio"
    ]
    .astype("string")
    .str.strip()
    .str.zfill(7)
)


comparacao_area = (
    brluc_soja_teste[
        [
            "Municipality code",
            "Municipality name",
            "uf",
            "area_conversao_candidata_ha"
        ]
    ]
    .rename(
        columns={
            "Municipality code":
                "codigo_ibge",

            "Municipality name":
                "municipio_original"
        }
    )
    .merge(
        area_equipe[
            [
                "codigo_ibge",
                "municipio",
                "area_conversao_ha"
            ]
        ],
        on="codigo_ibge",
        how="outer",
        indicator=True,
        validate="one_to_one"
    )
)


comparacao_area[
    "diferenca_area_ha"
] = (
    comparacao_area[
        "area_conversao_candidata_ha"
    ]
    -
    comparacao_area[
        "area_conversao_ha"
    ]
).abs()


print(
    "Status do merge:"
)

display(
    comparacao_area[
        "_merge"
    ]
    .value_counts()
)


print(
    "\nMaior diferença:"
)

print(
    comparacao_area[
        "diferenca_area_ha"
    ]
    .max()
)


print(
    "\nRegistros com diferença > 0.01 ha:"
)

print(
    (
        comparacao_area[
            "diferenca_area_ha"
        ]
        > 0.01
    )
    .sum()
)


display(
    comparacao_area[
        [
            "codigo_ibge",
            "municipio_original",
            "area_conversao_candidata_ha",
            "area_conversao_ha",
            "diferenca_area_ha"
        ]
    ]
    .head(10)
)

Status do merge:


_merge
both          1658
left_only        0
right_only       0
Name: count, dtype: int64


Maior diferença:
337608.7343036096

Registros com diferença > 0.01 ha:
1556


,codigo_ibge,municipio_original,area_conversao_candidata_ha,area_conversao_ha,diferenca_area_ha
0,4100103,Abatiá,4322.966534,371.85,3951.116534
1,4100202,Adrianópolis,36.205975,1.30,34.905975
2,4100301,Agudos do Sul,367.334087,1.29,366.044087
3,4100400,Almirante Tamandaré,649.244650,17.58,631.664650
4,4100459,Altamira do Paraná,764.064713,0.00,764.064713
5,4100509,Altônia,5326.875748,141.22,5185.655748
6,4100608,Alto Paraná,783.813306,4.86,778.953306
7,4100707,Alto Piquiri,24834.296071,4071.25,20763.046071
8,4100806,Alvorada do Sul,9696.450501,9311.82,384.630501
9,4100905,Amaporã,2275.543166,3.21,2272.333166


In [23]:
# ============================================================
# AUDITORIA — "ÁREA DE CONVERSÃO" DA EQUIPE × FROM SOYBEAN
# ============================================================

comparacao_area_soja = (
    brluc_soja_teste[
        [
            "Municipality code",
            "Municipality name",
            "uf",
            "from Soybean",
            "Area1_t1",
            "area_conversao_candidata_ha"
        ]
    ]
    .rename(
        columns={
            "Municipality code":
                "codigo_ibge",

            "Municipality name":
                "municipio_original"
        }
    )
    .merge(
        area_equipe[
            [
                "codigo_ibge",
                "area_conversao_ha"
            ]
        ],
        on="codigo_ibge",
        how="inner",
        validate="one_to_one"
    )
)


comparacao_area_soja[
    "dif_csv_vs_from_soybean"
] = (
    comparacao_area_soja[
        "area_conversao_ha"
    ]
    -
    comparacao_area_soja[
        "from Soybean"
    ]
).abs()


print(
    "Maior diferença:"
)

print(
    comparacao_area_soja[
        "dif_csv_vs_from_soybean"
    ].max()
)


print(
    "\nRegistros com diferença > 0.01 ha:"
)

print(
    (
        comparacao_area_soja[
            "dif_csv_vs_from_soybean"
        ]
        > 0.01
    )
    .sum()
)


display(
    comparacao_area_soja[
        [
            "codigo_ibge",
            "municipio_original",
            "from Soybean",
            "area_conversao_ha",
            "dif_csv_vs_from_soybean"
        ]
    ]
    .head(10)
)

Maior diferença:
0.004996087645963598

Registros com diferença > 0.01 ha:
0


,codigo_ibge,municipio_original,from Soybean,area_conversao_ha,dif_csv_vs_from_soybean
0,4100103,Abatiá,371.845266,371.85,0.004734
1,4100202,Adrianópolis,1.302738,1.30,0.002738
2,4100301,Agudos do Sul,1.287731,1.29,0.002269
3,4100400,Almirante Tamandaré,17.580535,17.58,0.000535
4,4100459,Altamira do Paraná,0.000000,0.00,0.000000
5,4100509,Altônia,141.222467,141.22,0.002467
6,4100608,Alto Paraná,4.863484,4.86,0.003484
7,4100707,Alto Piquiri,4071.245017,4071.25,0.004983
8,4100806,Alvorada do Sul,9311.821595,9311.82,0.001595
9,4100905,Amaporã,3.213496,3.21,0.003496


In [24]:
# ============================================================
# AUDITORIA — PERCENTUAL DE CONVERSÃO DA EQUIPE
# ============================================================

percentual_equipe = (
    bases_derivadas_brluc[
        "Percentual de conversão"
    ]
    .copy()
)


percentual_equipe[
    "codigo_ibge"
] = (
    percentual_equipe[
        "Codigo_Municipio"
    ]
    .astype("string")
    .str.strip()
    .str.zfill(7)
)


teste_percentual = (
    brluc_soja_teste[
        [
            "Municipality code",
            "Municipality name",
            "from Soybean",
            "Area1_t1",
            "Area1e2_t1",
            "area_conversao_candidata_ha"
        ]
    ]
    .rename(
        columns={
            "Municipality code":
                "codigo_ibge",

            "Municipality name":
                "municipio_original"
        }
    )
    .merge(
        percentual_equipe[
            [
                "codigo_ibge",
                "Percentual_Conversao"
            ]
        ],
        on="codigo_ibge",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Hipótese A:
# permanência soja / área soja em t1
# ------------------------------------------------------------

teste_percentual[
    "pct_from_soybean_area1"
] = (
    teste_percentual[
        "from Soybean"
    ]
    /
    teste_percentual[
        "Area1_t1"
    ]
    .replace(0, np.nan)
    * 100
)


# ------------------------------------------------------------
# Hipótese B:
# conversão de outras classes / área soja em t1
# ------------------------------------------------------------

teste_percentual[
    "pct_conversao_area1"
] = (
    teste_percentual[
        "area_conversao_candidata_ha"
    ]
    /
    teste_percentual[
        "Area1_t1"
    ]
    .replace(0, np.nan)
    * 100
)


# ------------------------------------------------------------
# Hipótese C:
# from Soybean / Area1e2_t1
# ------------------------------------------------------------

teste_percentual[
    "pct_from_soybean_area1e2"
] = (
    teste_percentual[
        "from Soybean"
    ]
    /
    teste_percentual[
        "Area1e2_t1"
    ]
    .replace(0, np.nan)
    * 100
)


for coluna in [
    "pct_from_soybean_area1",
    "pct_conversao_area1",
    "pct_from_soybean_area1e2"
]:

    diferenca = (
        teste_percentual[
            coluna
        ]
        -
        teste_percentual[
            "Percentual_Conversao"
        ]
    ).abs()

    print("\n" + "=" * 70)

    print(
        coluna
    )

    print(
        "Maior diferença:",
        diferenca.max()
    )

    print(
        "Diferenças > 0.01:",
        (diferenca > 0.01).sum()
    )


pct_from_soybean_area1
Maior diferença: 0.00499925997841677
Diferenças > 0.01: 0

pct_conversao_area1
Maior diferença: 100.00000000000048
Diferenças > 0.01: 1556

pct_from_soybean_area1e2
Maior diferença: 46.6803899756368
Diferenças > 0.01: 1142


In [25]:
# ============================================================
# AUDITORIA — TAXA DE EMISSÃO DA EQUIPE × BRLUC OFICIAL
# ============================================================

taxa_equipe = (
    bases_derivadas_brluc[
        "Taxa de emissão"
    ]
    .copy()
)


taxa_equipe[
    "codigo_ibge"
] = (
    taxa_equipe[
        "Codigo_Municipio"
    ]
    .astype("string")
    .str.strip()
    .str.zfill(7)
)


comparacao_taxa = (
    brluc_soja_oficial[
        [
            "Municipality code",
            "Municipality name",
            "Emission rates",
            "Emission rates_ci95low",
            "Emission rates_ci95upp"
        ]
    ]
    .rename(
        columns={
            "Municipality code":
                "codigo_ibge",

            "Municipality name":
                "municipio_original"
        }
    )
    .merge(
        taxa_equipe[
            [
                "codigo_ibge",
                "Taxa_Emissao",
                "IC_Inferior",
                "IC_Superior"
            ]
        ],
        on="codigo_ibge",
        how="outer",
        indicator=True,
        validate="one_to_one"
    )
)


comparacao_taxa[
    "dif_taxa"
] = (
    comparacao_taxa[
        "Emission rates"
    ]
    -
    comparacao_taxa[
        "Taxa_Emissao"
    ]
).abs()


comparacao_taxa[
    "dif_ic_inferior"
] = (
    comparacao_taxa[
        "Emission rates_ci95low"
    ]
    -
    comparacao_taxa[
        "IC_Inferior"
    ]
).abs()


comparacao_taxa[
    "dif_ic_superior"
] = (
    comparacao_taxa[
        "Emission rates_ci95upp"
    ]
    -
    comparacao_taxa[
        "IC_Superior"
    ]
).abs()


print(
    "Status do merge:"
)

display(
    comparacao_taxa[
        "_merge"
    ]
    .value_counts()
)


for coluna in [
    "dif_taxa",
    "dif_ic_inferior",
    "dif_ic_superior"
]:

    print(
        "\n",
        coluna,
        "máxima =",
        comparacao_taxa[
            coluna
        ].max()
    )

    print(
        "registros > 0.01 =",
        (
            comparacao_taxa[
                coluna
            ]
            > 0.01
        )
        .sum()
    )

Status do merge:


_merge
both          1658
left_only        0
right_only       0
Name: count, dtype: int64


 dif_taxa máxima = 0.004996250879599984
registros > 0.01 = 0

 dif_ic_inferior máxima = 0.0049932901690959985
registros > 0.01 = 0

 dif_ic_superior máxima = 0.004983897016832994
registros > 0.01 = 0


In [26]:
# ============================================================
# AUDITORIA TERRITORIAL — EMISSÃO ABSOLUTA DA EQUIPE
# ============================================================

emissao_equipe = (
    bases_derivadas_brluc[
        "Emissão absoluta"
    ]
    .copy()
)


# Padronizar código IBGE
emissao_equipe[
    "codigo_ibge"
] = (
    pd.to_numeric(
        emissao_equipe[
            "Codigo_Municipio"
        ],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(7)
)


# Universo oficial BRLUC — Brasil
codigos_oficiais_brasil = set(
    brluc_soja_original[
        "Municipality code"
    ]
    .dropna()
    .astype("string")
    .str.zfill(7)
)


codigos_equipe_brasil = set(
    emissao_equipe[
        "codigo_ibge"
    ]
    .dropna()
)


extras_equipe = (
    codigos_equipe_brasil
    -
    codigos_oficiais_brasil
)


faltantes_equipe = (
    codigos_oficiais_brasil
    -
    codigos_equipe_brasil
)


print("=" * 70)
print("AUDITORIA TERRITORIAL — EMISSÃO ABSOLUTA")
print("=" * 70)


print(
    "\nLinhas da equipe:",
    len(emissao_equipe)
)


print(
    "Códigos únicos da equipe:",
    emissao_equipe[
        "codigo_ibge"
    ].nunique()
)


print(
    "Códigos oficiais BRLUC:",
    len(
        codigos_oficiais_brasil
    )
)


print(
    "\nDuplicatas na equipe:"
)

print(
    emissao_equipe
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nCódigos extras na equipe:"
)

print(
    extras_equipe
)


print(
    "\nCódigos oficiais ausentes na equipe:"
)

print(
    faltantes_equipe
)


if extras_equipe:

    print(
        "\nRegistros extras:"
    )

    display(
        emissao_equipe[
            emissao_equipe[
                "codigo_ibge"
            ]
            .isin(
                extras_equipe
            )
        ]
    )

AUDITORIA TERRITORIAL — EMISSÃO ABSOLUTA

Linhas da equipe: 5570
Códigos únicos da equipe: 5570
Códigos oficiais BRLUC: 5569

Duplicatas na equipe:
0

Códigos extras na equipe:
{'2605459'}

Códigos oficiais ausentes na equipe:
set()

Registros extras:


,Codigo_Municipio,Municipio,Estado,Emissao_Absoluta,IC_Inferior,IC_Superior,codigo_ibge
1525,2605459,Fernando de Noronha,Pernambuco,0.0,NaN,NaN,2605459


In [27]:
# ============================================================
# AUDITORIA — EMISSÃO ABSOLUTA E IC95%
# ============================================================

emissao_oficial_brasil = (
    brluc_soja_original[
        [
            "Municipality code",
            "Municipality name",
            "Absolute emissions",
            "Absolute emissions_ci95low",
            "Absolute emissions_ci95upp"
        ]
    ]
    .rename(
        columns={
            "Municipality code":
                "codigo_ibge",

            "Municipality name":
                "municipio_original"
        }
    )
    .copy()
)


emissao_oficial_brasil[
    "codigo_ibge"
] = (
    emissao_oficial_brasil[
        "codigo_ibge"
    ]
    .astype("string")
    .str.zfill(7)
)


comparacao_emissao = (
    emissao_oficial_brasil
    .merge(
        emissao_equipe[
            [
                "codigo_ibge",
                "Municipio",
                "Emissao_Absoluta",
                "IC_Inferior",
                "IC_Superior"
            ]
        ],
        on="codigo_ibge",
        how="inner",
        validate="one_to_one"
    )
)


comparacao_emissao[
    "dif_emissao"
] = (
    comparacao_emissao[
        "Absolute emissions"
    ]
    -
    comparacao_emissao[
        "Emissao_Absoluta"
    ]
).abs()


comparacao_emissao[
    "dif_ic_inferior"
] = (
    comparacao_emissao[
        "Absolute emissions_ci95low"
    ]
    -
    comparacao_emissao[
        "IC_Inferior"
    ]
).abs()


comparacao_emissao[
    "dif_ic_superior"
] = (
    comparacao_emissao[
        "Absolute emissions_ci95upp"
    ]
    -
    comparacao_emissao[
        "IC_Superior"
    ]
).abs()


print(
    "Registros comparados:",
    len(
        comparacao_emissao
    )
)


for coluna in [
    "dif_emissao",
    "dif_ic_inferior",
    "dif_ic_superior"
]:

    print(
        "\n" + coluna
    )

    print(
        "Maior diferença:",
        comparacao_emissao[
            coluna
        ].max()
    )

    print(
        "Registros > 0.01:",
        (
            comparacao_emissao[
                coluna
            ]
            > 0.01
        )
        .sum()
    )


print(
    "\nNulos — IC inferior:"
)

print(
    "Oficial:",
    comparacao_emissao[
        "Absolute emissions_ci95low"
    ]
    .isna()
    .sum()
)

print(
    "Equipe:",
    comparacao_emissao[
        "IC_Inferior"
    ]
    .isna()
    .sum()
)


print(
    "\nNulos — IC superior:"
)

print(
    "Oficial:",
    comparacao_emissao[
        "Absolute emissions_ci95upp"
    ]
    .isna()
    .sum()
)

print(
    "Equipe:",
    comparacao_emissao[
        "IC_Superior"
    ]
    .isna()
    .sum()
)

Registros comparados: 5569

dif_emissao
Maior diferença: 0.004998296143839953
Registros > 0.01: 0

dif_ic_inferior
Maior diferença: 0.004997404302002906
Registros > 0.01: 0

dif_ic_superior
Maior diferença: 0.00499948269862216
Registros > 0.01: 0

Nulos — IC inferior:
Oficial: 2930
Equipe: 0

Nulos — IC superior:
Oficial: 2930
Equipe: 0


In [28]:
# ============================================================
# VARIÁVEIS DERIVADAS — PERMANÊNCIA E CONVERSÃO PARA SOJA
# ============================================================

brluc_soja_reconstruido = (
    brluc_soja_oficial
    .copy()
)


# ------------------------------------------------------------
# Área que já era soja em t0 e continua soja em t1
# ------------------------------------------------------------

brluc_soja_reconstruido[
    "area_soja_persistente_2000_2019_ha"
] = (
    brluc_soja_reconstruido[
        "from Soybean"
    ]
)


# ------------------------------------------------------------
# Área proveniente de outras classes e convertida para soja
# ------------------------------------------------------------

brluc_soja_reconstruido[
    "area_conversao_para_soja_2000_2019_ha"
] = (
    brluc_soja_reconstruido[
        "Area1_t1"
    ]
    -
    brluc_soja_reconstruido[
        "from Soybean"
    ]
)


# ------------------------------------------------------------
# Percentual de permanência
# ------------------------------------------------------------

brluc_soja_reconstruido[
    "percentual_persistencia_soja_pct"
] = np.where(
    brluc_soja_reconstruido[
        "Area1_t1"
    ] > 0,

    (
        brluc_soja_reconstruido[
            "from Soybean"
        ]
        /
        brluc_soja_reconstruido[
            "Area1_t1"
        ]
        * 100
    ),

    np.nan
)


# ------------------------------------------------------------
# Percentual convertido de outras classes para soja
# ------------------------------------------------------------

brluc_soja_reconstruido[
    "percentual_conversao_para_soja_pct"
] = np.where(
    brluc_soja_reconstruido[
        "Area1_t1"
    ] > 0,

    (
        brluc_soja_reconstruido[
            "area_conversao_para_soja_2000_2019_ha"
        ]
        /
        brluc_soja_reconstruido[
            "Area1_t1"
        ]
        * 100
    ),

    np.nan
)


# ------------------------------------------------------------
# Controle: permanência + conversão deve ser 100%
# quando existe área de soja
# ------------------------------------------------------------

brluc_soja_reconstruido[
    "controle_pct"
] = (
    brluc_soja_reconstruido[
        "percentual_persistencia_soja_pct"
    ]
    +
    brluc_soja_reconstruido[
        "percentual_conversao_para_soja_pct"
    ]
)


print(
    "Maior desvio de 100%:"
)

print(
    (
        brluc_soja_reconstruido[
            "controle_pct"
        ]
        .dropna()
        -
        100
    )
    .abs()
    .max()
)


print(
    "\nMunicípios sem área de soja em Area1_t1:"
)

print(
    (
        brluc_soja_reconstruido[
            "Area1_t1"
        ]
        == 0
    )
    .sum()
)


display(
    brluc_soja_reconstruido[
        [
            "Municipality code",
            "Municipality name",
            "uf",

            "Area1_t1",

            "area_soja_persistente_2000_2019_ha",
            "area_conversao_para_soja_2000_2019_ha",

            "percentual_persistencia_soja_pct",
            "percentual_conversao_para_soja_pct",

            "controle_pct"
        ]
    ]
    .head(10)
)

Maior desvio de 100%:
2.842170943040401e-14

Municípios sem área de soja em Area1_t1:
102


,Municipality code,Municipality name,uf,Area1_t1,area_soja_persistente_2000_2019_ha,area_conversao_para_soja_2000_2019_ha,percentual_persistencia_soja_pct,percentual_conversao_para_soja_pct,controle_pct
9480,4100103,Abatiá,PR,4694.811800,371.845266,4322.966534,7.920344,92.079656,100.0
9481,4100202,Adrianópolis,PR,37.508712,1.302738,36.205975,3.473161,96.526839,100.0
9482,4100301,Agudos do Sul,PR,368.621819,1.287731,367.334087,0.349337,99.650663,100.0
9483,4100400,Almirante Tamandaré,PR,666.825185,17.580535,649.244650,2.636453,97.363547,100.0
9484,4100459,Altamira do Paraná,PR,764.064713,0.000000,764.064713,0.000000,100.000000,100.0
9485,4100509,Altônia,PR,5468.098215,141.222467,5326.875748,2.582661,97.417339,100.0
9486,4100608,Alto Paraná,PR,788.676790,4.863484,783.813306,0.616664,99.383336,100.0
9487,4100707,Alto Piquiri,PR,28905.541088,4071.245017,24834.296071,14.084653,85.915347,100.0
9488,4100806,Alvorada do Sul,PR,19008.272096,9311.821595,9696.450501,48.988259,51.011741,100.0
9489,4100905,Amaporã,PR,2278.756662,3.213496,2275.543166,0.141020,99.858980,100.0


In [29]:
# ============================================================
# AUDITORIA DOS ICs AUSENTES — EMISSÃO ABSOLUTA
# ============================================================

casos_ic_ausente = (
    comparacao_emissao[
        comparacao_emissao[
            "Absolute emissions_ci95low"
        ]
        .isna()
    ]
    .copy()
)


print(
    "Casos em que o IC é ausente na fonte oficial:"
)

print(
    len(casos_ic_ausente)
)


print(
    "\nNulos no IC inferior da equipe nesses casos:"
)

print(
    casos_ic_ausente[
        "IC_Inferior"
    ]
    .isna()
    .sum()
)


print(
    "\nNulos no IC superior da equipe nesses casos:"
)

print(
    casos_ic_ausente[
        "IC_Superior"
    ]
    .isna()
    .sum()
)


print(
    "\nValores mais frequentes no IC inferior da equipe:"
)

display(
    casos_ic_ausente[
        "IC_Inferior"
    ]
    .value_counts(
        dropna=False
    )
    .head(10)
)


print(
    "\nValores mais frequentes no IC superior da equipe:"
)

display(
    casos_ic_ausente[
        "IC_Superior"
    ]
    .value_counts(
        dropna=False
    )
    .head(10)
)


display(
    casos_ic_ausente[
        [
            "codigo_ibge",
            "municipio_original",
            "Absolute emissions",
            "Absolute emissions_ci95low",
            "Absolute emissions_ci95upp",
            "Emissao_Absoluta",
            "IC_Inferior",
            "IC_Superior"
        ]
    ]
    .head(15)
)    

Casos em que o IC é ausente na fonte oficial:
2930

Nulos no IC inferior da equipe nesses casos:
0

Nulos no IC superior da equipe nesses casos:
0

Valores mais frequentes no IC inferior da equipe:


IC_Inferior
0.0    2930
Name: count, dtype: int64


Valores mais frequentes no IC superior da equipe:


IC_Superior
0.0    2930
Name: count, dtype: int64

,codigo_ibge,municipio_original,Absolute emissions,Absolute emissions_ci95low,Absolute emissions_ci95upp,Emissao_Absoluta,IC_Inferior,IC_Superior
9,1100106,Guajará-Mirim,0.0,NaN,NaN,0.0,0.0,0.0
29,1100601,Cacaulândia,0.0,NaN,NaN,0.0,0.0,0.0
35,1101005,Governador Jorge Teixeira,0.0,NaN,NaN,0.0,0.0,0.0
37,1101203,Ministro Andreazza,0.0,NaN,NaN,0.0,0.0,0.0
39,1101401,Monte Negro,0.0,NaN,NaN,0.0,0.0,0.0
40,1101435,Nova União,0.0,NaN,NaN,0.0,0.0,0.0
52,1200013,Acrelândia,0.0,NaN,NaN,0.0,0.0,0.0
53,1200054,Assis Brasil,0.0,NaN,NaN,0.0,0.0,0.0
54,1200104,Brasiléia,0.0,NaN,NaN,0.0,0.0,0.0
55,1200138,Bujari,0.0,NaN,NaN,0.0,0.0,0.0


In [30]:
# ============================================================
# AUDITORIA DOS ICs AUSENTES — TAXA DE EMISSÃO
# ============================================================

print(
    "Nulos oficiais — IC inferior da taxa:"
)

print(
    comparacao_taxa[
        "Emission rates_ci95low"
    ]
    .isna()
    .sum()
)


print(
    "\nNulos equipe — IC inferior:"
)

print(
    comparacao_taxa[
        "IC_Inferior"
    ]
    .isna()
    .sum()
)


print(
    "\nNulos oficiais — IC superior da taxa:"
)

print(
    comparacao_taxa[
        "Emission rates_ci95upp"
    ]
    .isna()
    .sum()
)


print(
    "\nNulos equipe — IC superior:"
)

print(
    comparacao_taxa[
        "IC_Superior"
    ]
    .isna()
    .sum()
)


casos_taxa_ic_ausente = (
    comparacao_taxa[
        comparacao_taxa[
            "Emission rates_ci95low"
        ]
        .isna()
    ]
    .copy()
)


print(
    "\nQuantidade de casos oficiais sem IC:"
)

print(
    len(
        casos_taxa_ic_ausente
    )
)


display(
    casos_taxa_ic_ausente[
        [
            "codigo_ibge",
            "municipio_original",
            "Emission rates",
            "Emission rates_ci95low",
            "Emission rates_ci95upp",
            "Taxa_Emissao",
            "IC_Inferior",
            "IC_Superior"
        ]
    ]
    .head(15)
)

Nulos oficiais — IC inferior da taxa:
118

Nulos equipe — IC inferior:
0

Nulos oficiais — IC superior da taxa:
118

Nulos equipe — IC superior:
0

Quantidade de casos oficiais sem IC:
118


,codigo_ibge,municipio_original,Emission rates,Emission rates_ci95low,Emission rates_ci95upp,Taxa_Emissao,IC_Inferior,IC_Superior
135,4109500,Guaraqueçaba,0.0,NaN,NaN,0.0,0.0,0.0
215,4115705,Matinhos,0.0,NaN,NaN,0.0,0.0,0.0
275,4119954,Pontal do Paraná,0.0,NaN,NaN,0.0,0.0,0.0
406,4200606,Águas Mornas,0.0,NaN,NaN,0.0,0.0,0.0
414,4201257,Apiúna,0.0,NaN,NaN,0.0,0.0,0.0
421,4201703,Ascurra,0.0,NaN,NaN,0.0,0.0,0.0
424,4201950,Balneário Arroio do Silva,0.0,NaN,NaN,0.0,0.0,0.0
425,4202008,Balneário Camboriú,0.0,NaN,NaN,0.0,0.0,0.0
426,4202057,Balneário Barra do Sul,0.0,NaN,NaN,0.0,0.0,0.0
427,4202073,Balneário Gaivota,0.0,NaN,NaN,0.0,0.0,0.0


In [31]:
# ============================================================
# AUDITORIA — VARIÁVEIS RECONSTRUÍDAS
# ============================================================

variaveis_reconstruidas = [
    "area_soja_persistente_2000_2019_ha",
    "area_conversao_para_soja_2000_2019_ha",
    "percentual_persistencia_soja_pct",
    "percentual_conversao_para_soja_pct"
]


print("=" * 70)
print("AUDITORIA DAS VARIÁVEIS RECONSTRUÍDAS")
print("=" * 70)


for coluna in variaveis_reconstruidas:

    print(
        f"\n{coluna}"
    )

    print(
        "Nulos:",
        brluc_soja_reconstruido[
            coluna
        ]
        .isna()
        .sum()
    )

    print(
        "Negativos:",
        (
            brluc_soja_reconstruido[
                coluna
            ]
            < 0
        )
        .sum()
    )

    print(
        "Mínimo:",
        brluc_soja_reconstruido[
            coluna
        ]
        .min()
    )

    print(
        "Máximo:",
        brluc_soja_reconstruido[
            coluna
        ]
        .max()
    )


print(
    "\nPercentuais de conversão > 100:"
)

print(
    (
        brluc_soja_reconstruido[
            "percentual_conversao_para_soja_pct"
        ]
        > 100
    )
    .sum()
)


print(
    "\nPercentuais de persistência > 100:"
)

print(
    (
        brluc_soja_reconstruido[
            "percentual_persistencia_soja_pct"
        ]
        > 100
    )
    .sum()
)


print(
    "\nMunicípios com Area1_t1 = 0:"
)

print(
    (
        brluc_soja_reconstruido[
            "Area1_t1"
        ]
        == 0
    )
    .sum()
)

AUDITORIA DAS VARIÁVEIS RECONSTRUÍDAS

area_soja_persistente_2000_2019_ha
Nulos: 0
Negativos: 0
Mínimo: 0.0
Máximo: 359978.833949408

area_conversao_para_soja_2000_2019_ha
Nulos: 0
Negativos: 0
Mínimo: 0.0
Máximo: 355001.6343036098

percentual_persistencia_soja_pct
Nulos: 102
Negativos: 0
Mínimo: 0.0
Máximo: 100.0

percentual_conversao_para_soja_pct
Nulos: 102
Negativos: 0
Mínimo: 0.0
Máximo: 100.0

Percentuais de conversão > 100:
0

Percentuais de persistência > 100:
0

Municípios com Area1_t1 = 0:
102


## Resultado da auditoria dos arquivos derivados da equipe

A comparação dos arquivos derivados com o Appendix C original do BRLUC
identificou que:

- a taxa de emissão e seus intervalos de confiança correspondem aos
  valores oficiais, com arredondamento para duas casas decimais;
- a emissão absoluta também corresponde aos valores oficiais;
- o arquivo denominado `area_conversao` corresponde, na realidade,
  à variável `from Soybean`, representando permanência da classe
  soja entre t0 e t1;
- o denominado `Percentual_Conversao` corresponde a
  `from Soybean / Area1_t1 × 100`, portanto representa percentual
  de persistência da soja, e não percentual convertido para soja;
- valores ausentes de intervalos de confiança foram convertidos para
  zero nos arquivos derivados, alterando o significado original;
- foi acrescentado o código 2605459 (Fernando de Noronha/PE),
  inexistente nos 5.569 registros oficiais da categoria Soybean.

Por esses motivos, a camada Curated do projeto será reconstruída
diretamente a partir do Appendix C original do BRLUC, preservando
valores ausentes e utilizando nomes semanticamente adequados.

O BRLUC será tratado como uma camada municipal estrutural referente
à transição t0 = 2000 e t1 = 2019. Não será criada artificialmente
uma dimensão anual 2019–2024.

In [32]:
# ============================================================
# CONSTRUÇÃO DA CAMADA CURATED — BRLUC SOJA
# ============================================================

brluc_curated = (
    brluc_soja_reconstruido
    .copy()
)


# ------------------------------------------------------------
# Renomear variáveis oficiais
# ------------------------------------------------------------

brluc_curated = (
    brluc_curated
    .rename(
        columns={
            "Municipality code":
                "codigo_ibge",

            "Municipality name":
                "municipio",

            "State name":
                "estado",

            "from Temporary":
                "area_origem_temporaria_ha",

            "from Soybean":
                "area_origem_soja_ha",

            "from Sugarcane":
                "area_origem_cana_ha",

            "from Permanent":
                "area_origem_cultura_permanente_ha",

            "from Pasture":
                "area_origem_pastagem_ha",

            "from Forestry":
                "area_origem_floresta_plantada_ha",

            "from Natural":
                "area_origem_natural_ha",

            "Area1_t1":
                "area_soja_t1_brluc_ha",

            "Area1e2_t1":
                "area_referencia_taxa_emissao_ha",

            "Absolute emissions":
                "emissao_absoluta_co2_t_ano",

            "Absolute emissions_sd":
                "emissao_absoluta_co2_se",

            "Absolute emissions_ci95low":
                "emissao_absoluta_co2_ic95_inf",

            "Absolute emissions_ci95upp":
                "emissao_absoluta_co2_ic95_sup",

            "Emission rates":
                "taxa_emissao_co2_t_ha_ano",

            "Emission rates_sd":
                "taxa_emissao_co2_se",

            "Emission rates_ci95low":
                "taxa_emissao_co2_ic95_inf",

            "Emission rates_ci95upp":
                "taxa_emissao_co2_ic95_sup"
        }
    )
)


# ------------------------------------------------------------
# Região
# ------------------------------------------------------------

MAPA_REGIAO = {
    "DF": "Centro-Oeste",
    "GO": "Centro-Oeste",
    "MS": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "PR": "Sul",
    "RS": "Sul",
    "SC": "Sul"
}


brluc_curated[
    "regiao"
] = (
    brluc_curated[
        "uf"
    ]
    .map(
        MAPA_REGIAO
    )
)


# ------------------------------------------------------------
# Metadados da camada
# ------------------------------------------------------------

brluc_curated[
    "cultura"
] = "Soja"


brluc_curated[
    "periodo_inicio_brluc"
] = 2000


brluc_curated[
    "periodo_fim_brluc"
] = 2019


brluc_curated[
    "fonte"
] = "Embrapa BRLUC"


brluc_curated[
    "versao_fonte"
] = "BRLUC 2.1"


# ------------------------------------------------------------
# Flags de disponibilidade de incerteza
# ------------------------------------------------------------

brluc_curated[
    "ic95_emissao_absoluta_disponivel"
] = (
    brluc_curated[
        "emissao_absoluta_co2_ic95_inf"
    ].notna()
    &
    brluc_curated[
        "emissao_absoluta_co2_ic95_sup"
    ].notna()
)


brluc_curated[
    "ic95_taxa_emissao_disponivel"
] = (
    brluc_curated[
        "taxa_emissao_co2_ic95_inf"
    ].notna()
    &
    brluc_curated[
        "taxa_emissao_co2_ic95_sup"
    ].notna()
)


# ------------------------------------------------------------
# Seleção e ordem final das colunas
# ------------------------------------------------------------

COLUNAS_CURATED = [
    "codigo_ibge",
    "municipio",
    "estado",
    "uf",
    "regiao",
    "cultura",

    "periodo_inicio_brluc",
    "periodo_fim_brluc",

    "area_origem_temporaria_ha",
    "area_origem_soja_ha",
    "area_origem_cana_ha",
    "area_origem_cultura_permanente_ha",
    "area_origem_pastagem_ha",
    "area_origem_floresta_plantada_ha",
    "area_origem_natural_ha",

    "area_soja_t1_brluc_ha",
    "area_soja_persistente_2000_2019_ha",
    "area_conversao_para_soja_2000_2019_ha",

    "percentual_persistencia_soja_pct",
    "percentual_conversao_para_soja_pct",

    "area_referencia_taxa_emissao_ha",

    "emissao_absoluta_co2_t_ano",
    "emissao_absoluta_co2_se",
    "emissao_absoluta_co2_ic95_inf",
    "emissao_absoluta_co2_ic95_sup",
    "ic95_emissao_absoluta_disponivel",

    "taxa_emissao_co2_t_ha_ano",
    "taxa_emissao_co2_se",
    "taxa_emissao_co2_ic95_inf",
    "taxa_emissao_co2_ic95_sup",
    "ic95_taxa_emissao_disponivel",

    "fonte",
    "versao_fonte"
]


brluc_curated = (
    brluc_curated[
        COLUNAS_CURATED
    ]
    .copy()
)


print(
    "Dimensão da Curated:"
)

print(
    brluc_curated.shape
)


display(
    brluc_curated.head()
)

Dimensão da Curated:
(1658, 33)


,codigo_ibge,municipio,estado,uf,regiao,cultura,periodo_inicio_brluc,periodo_fim_brluc,area_origem_temporaria_ha,area_origem_soja_ha,...,emissao_absoluta_co2_ic95_inf,emissao_absoluta_co2_ic95_sup,ic95_emissao_absoluta_disponivel,taxa_emissao_co2_t_ha_ano,taxa_emissao_co2_se,taxa_emissao_co2_ic95_inf,taxa_emissao_co2_ic95_sup,ic95_taxa_emissao_disponivel,fonte,versao_fonte
9480,4100103,Abatiá,Paraná,PR,Sul,Soja,2000,2019,2488.168493,371.845266,...,3900.076021,6125.910447,True,0.538510,0.060996,0.418957,0.658063,True,Embrapa BRLUC,BRLUC 2.1
9481,4100202,Adrianópolis,Paraná,PR,Sul,Soja,2000,2019,1.450002,1.302738,...,514.475181,812.405761,True,15.502237,1.775912,12.021450,18.983024,True,Embrapa BRLUC,BRLUC 2.1
9482,4100301,Agudos do Sul,Paraná,PR,Sul,Soja,2000,2019,326.200765,1.287731,...,693.892581,883.791107,True,2.093843,0.128585,1.841816,2.345869,True,Embrapa BRLUC,BRLUC 2.1
9483,4100400,Almirante Tamandaré,Paraná,PR,Sul,Soja,2000,2019,508.821336,17.580535,...,2301.492278,3703.271869,True,4.428772,0.527486,3.394899,5.462644,True,Embrapa BRLUC,BRLUC 2.1
9484,4100459,Altamira do Paraná,Paraná,PR,Sul,Soja,2000,2019,138.539817,0.000000,...,1443.070473,2287.227043,True,1.921486,0.221851,1.486659,2.356313,True,Embrapa BRLUC,BRLUC 2.1


In [33]:
# ============================================================
# VALIDAÇÃO FINAL — CAMADA CURATED BRLUC
# ============================================================

print("=" * 70)
print("VALIDAÇÃO FINAL — BRLUC CURATED")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    brluc_curated.shape
)


print(
    "\nMunicípios distintos:"
)

print(
    brluc_curated[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas de código IBGE:"
)

print(
    brluc_curated
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nDistribuição por UF:"
)

display(
    brluc_curated[
        "uf"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nPeríodo BRLUC:"
)

print(
    "Início:",
    brluc_curated[
        "periodo_inicio_brluc"
    ]
    .unique()
)

print(
    "Fim:",
    brluc_curated[
        "periodo_fim_brluc"
    ]
    .unique()
)


print(
    "\nConversão negativa:"
)

print(
    (
        brluc_curated[
            "area_conversao_para_soja_2000_2019_ha"
        ]
        < 0
    )
    .sum()
)


print(
    "\nPercentual conversão fora de 0–100:"
)

print(
    (
        (
            brluc_curated[
                "percentual_conversao_para_soja_pct"
            ]
            < 0
        )
        |
        (
            brluc_curated[
                "percentual_conversao_para_soja_pct"
            ]
            > 100
        )
    )
    .sum()
)


print(
    "\nEmissões absolutas negativas:"
)

print(
    (
        brluc_curated[
            "emissao_absoluta_co2_t_ano"
        ]
        < 0
    )
    .sum()
)


print(
    "\nTaxas de emissão negativas:"
)

print(
    (
        brluc_curated[
            "taxa_emissao_co2_t_ha_ano"
        ]
        < 0
    )
    .sum()
)


print(
    "\nPercentuais NULL:"
)

print(
    brluc_curated[
        "percentual_conversao_para_soja_pct"
    ]
    .isna()
    .sum()
)


print(
    "\nIC95 emissão absoluta indisponível:"
)

print(
    (
        ~brluc_curated[
            "ic95_emissao_absoluta_disponivel"
        ]
    )
    .sum()
)


print(
    "\nIC95 taxa de emissão indisponível:"
)

print(
    (
        ~brluc_curated[
            "ic95_taxa_emissao_disponivel"
        ]
    )
    .sum()
)    

VALIDAÇÃO FINAL — BRLUC CURATED

Dimensão:
(1658, 33)

Municípios distintos:
1658

Duplicatas de código IBGE:
0

Distribuição por UF:


uf
DF      1
GO    246
MS     79
MT    141
PR    399
RS    497
SC    295
Name: count, dtype: int64


Período BRLUC:
Início: [2000]
Fim: [2019]

Conversão negativa:
0

Percentual conversão fora de 0–100:
0

Emissões absolutas negativas:
17

Taxas de emissão negativas:
17

Percentuais NULL:
102

IC95 emissão absoluta indisponível:
102

IC95 taxa de emissão indisponível:
118


In [34]:
# ============================================================
# CONTROLES DE CONSISTÊNCIA MATEMÁTICA
# ============================================================

# ------------------------------------------------------------
# Origem das áreas deve recompor a área da soja em t1
# ------------------------------------------------------------

soma_origens_curated = (
    brluc_curated[
        [
            "area_origem_temporaria_ha",
            "area_origem_soja_ha",
            "area_origem_cana_ha",
            "area_origem_cultura_permanente_ha",
            "area_origem_pastagem_ha",
            "area_origem_floresta_plantada_ha",
            "area_origem_natural_ha"
        ]
    ]
    .sum(axis=1)
)


dif_area = (
    soma_origens_curated
    -
    brluc_curated[
        "area_soja_t1_brluc_ha"
    ]
).abs()


# ------------------------------------------------------------
# Persistência + conversão deve recompor a área t1
# ------------------------------------------------------------

dif_persistencia_conversao = (
    (
        brluc_curated[
            "area_soja_persistente_2000_2019_ha"
        ]
        +
        brluc_curated[
            "area_conversao_para_soja_2000_2019_ha"
        ]
    )
    -
    brluc_curated[
        "area_soja_t1_brluc_ha"
    ]
).abs()


# ------------------------------------------------------------
# Emissão / área de referência deve recompor taxa
# ------------------------------------------------------------

taxa_recalculada = (
    brluc_curated[
        "emissao_absoluta_co2_t_ano"
    ]
    /
    brluc_curated[
        "area_referencia_taxa_emissao_ha"
    ]
    .replace(
        0,
        np.nan
    )
)


dif_taxa = (
    taxa_recalculada
    -
    brluc_curated[
        "taxa_emissao_co2_t_ha_ano"
    ]
).abs()


print(
    "Maior diferença — soma origens × área t1:"
)

print(
    dif_area.max()
)


print(
    "\nMaior diferença — persistência + conversão × área t1:"
)

print(
    dif_persistencia_conversao.max()
)


print(
    "\nMaior diferença — emissão / área × taxa:"
)

print(
    dif_taxa.max()
)


print(
    "\nFalhas > 0.000001:"
)

print(
    "Área:",
    (dif_area > 0.000001).sum()
)

print(
    "Persistência/conversão:",
    (
        dif_persistencia_conversao
        > 0.000001
    ).sum()
)

print(
    "Taxa:",
    (
        dif_taxa
        > 0.000001
    ).sum()
)

Maior diferença — soma origens × área t1:
9.313225746154785e-10

Maior diferença — persistência + conversão × área t1:
2.9103830456733704e-11

Maior diferença — emissão / área × taxa:
9.592326932761353e-14

Falhas > 0.000001:
Área: 0
Persistência/conversão: 0
Taxa: 0


In [35]:
# ============================================================
# EXPORTAÇÃO — BRLUC CURATED
# ============================================================

BRLUC_CURATED_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "embrapa_brluc"
    / "municipio"
)


BRLUC_CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ARQUIVO_BRLUC_CURATED = (
    BRLUC_CURATED_DIR
    / "brluc_soja_municipio_centro_oeste_sul_2000_2019.csv"
)


# ------------------------------------------------------------
# Ordenação e índice limpo
# ------------------------------------------------------------

brluc_curated_exportacao = (
    brluc_curated
    .sort_values(
        by=[
            "uf",
            "codigo_ibge"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Exportação
# ------------------------------------------------------------

brluc_curated_exportacao.to_csv(
    ARQUIVO_BRLUC_CURATED,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 70)
print("EXPORTAÇÃO CONCLUÍDA")
print("=" * 70)

print(
    "\nArquivo:"
)

print(
    ARQUIVO_BRLUC_CURATED
)


print(
    "\nExiste:"
)

print(
    ARQUIVO_BRLUC_CURATED.exists()
)


print(
    "\nTamanho:"
)

print(
    f"{ARQUIVO_BRLUC_CURATED.stat().st_size / (1024 * 1024):.2f} MB"
)


print(
    "\nDimensão exportada:"
)

print(
    brluc_curated_exportacao.shape
)

EXPORTAÇÃO CONCLUÍDA

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\embrapa_brluc\municipio\brluc_soja_municipio_centro_oeste_sul_2000_2019.csv

Existe:
True

Tamanho:
0.64 MB

Dimensão exportada:
(1658, 33)


In [36]:
# ============================================================
# VALIDAÇÃO PÓS-EXPORTAÇÃO
# ============================================================

brluc_curated_recarregado = pd.read_csv(
    ARQUIVO_BRLUC_CURATED,
    dtype={
        "codigo_ibge": "string"
    },
    encoding="utf-8-sig"
)


brluc_curated_recarregado[
    "codigo_ibge"
] = (
    brluc_curated_recarregado[
        "codigo_ibge"
    ]
    .str.zfill(7)
)


print("=" * 70)
print("VALIDAÇÃO DO ARQUIVO EXPORTADO")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    brluc_curated_recarregado.shape
)


print(
    "\nMunicípios distintos:"
)

print(
    brluc_curated_recarregado[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    brluc_curated_recarregado
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nPercentuais de conversão NULL:"
)

print(
    brluc_curated_recarregado[
        "percentual_conversao_para_soja_pct"
    ]
    .isna()
    .sum()
)


print(
    "\nIC95 emissão absoluta inferior NULL:"
)

print(
    brluc_curated_recarregado[
        "emissao_absoluta_co2_ic95_inf"
    ]
    .isna()
    .sum()
)


print(
    "\nIC95 taxa de emissão inferior NULL:"
)

print(
    brluc_curated_recarregado[
        "taxa_emissao_co2_ic95_inf"
    ]
    .isna()
    .sum()
)


print(
    "\nEmissões negativas:"
)

print(
    (
        brluc_curated_recarregado[
            "emissao_absoluta_co2_t_ano"
        ]
        < 0
    )
    .sum()
)


print(
    "\nTaxas negativas:"
)

print(
    (
        brluc_curated_recarregado[
            "taxa_emissao_co2_t_ha_ano"
        ]
        < 0
    )
    .sum()
)


print(
    "\nCódigos IBGE iguais ao DataFrame original:"
)

print(
    set(
        brluc_curated_recarregado[
            "codigo_ibge"
        ]
    )
    ==
    set(
        brluc_curated_exportacao[
            "codigo_ibge"
        ]
    )
)

VALIDAÇÃO DO ARQUIVO EXPORTADO

Dimensão:
(1658, 33)

Municípios distintos:
1658

Duplicatas:
0

Percentuais de conversão NULL:
102

IC95 emissão absoluta inferior NULL:
102

IC95 taxa de emissão inferior NULL:
118

Emissões negativas:
17

Taxas negativas:
17

Códigos IBGE iguais ao DataFrame original:
True


In [37]:
# ============================================================
# CONTROLE NUMÉRICO PÓS-EXPORTAÇÃO
# ============================================================

COLUNAS_CONTROLE_EXPORTACAO = [
    "area_soja_t1_brluc_ha",
    "area_soja_persistente_2000_2019_ha",
    "area_conversao_para_soja_2000_2019_ha",
    "percentual_persistencia_soja_pct",
    "percentual_conversao_para_soja_pct",
    "emissao_absoluta_co2_t_ano",
    "taxa_emissao_co2_t_ha_ano"
]


original_controle = (
    brluc_curated_exportacao[
        [
            "codigo_ibge"
        ]
        +
        COLUNAS_CONTROLE_EXPORTACAO
    ]
    .set_index(
        "codigo_ibge"
    )
)


recarregado_controle = (
    brluc_curated_recarregado[
        [
            "codigo_ibge"
        ]
        +
        COLUNAS_CONTROLE_EXPORTACAO
    ]
    .set_index(
        "codigo_ibge"
    )
)


print("=" * 70)
print("DIFERENÇAS PÓS-EXPORTAÇÃO")
print("=" * 70)


for coluna in COLUNAS_CONTROLE_EXPORTACAO:

    diferenca = (
        original_controle[
            coluna
        ]
        -
        recarregado_controle[
            coluna
        ]
    ).abs()

    print(
        f"{coluna}:",
        diferenca.max()
    )

DIFERENÇAS PÓS-EXPORTAÇÃO
area_soja_t1_brluc_ha: 0.0
area_soja_persistente_2000_2019_ha: 0.0
area_conversao_para_soja_2000_2019_ha: 2.9103830456733704e-11
percentual_persistencia_soja_pct: 1.4210854715202004e-14
percentual_conversao_para_soja_pct: 1.4210854715202004e-14
emissao_absoluta_co2_t_ano: 6.071532165918825e-17
taxa_emissao_co2_t_ha_ano: 9.020562075079397e-17


## Conclusão

A camada BRLUC para soja foi reconstruída diretamente a partir do
Appendix C original da fonte Embrapa/BRLUC.

O recorte final contém 1.658 municípios das regiões Centro-Oeste e Sul,
sem duplicidade de código IBGE.

A análise utiliza a transição de uso da terra entre t0 = 2000 e
t1 = 2019. Os resultados do BRLUC não foram transformados em uma
série anual 2019–2024.

Foram preservados:

- áreas de origem das diferentes classes de uso da terra;
- área atribuída à soja em t1;
- permanência soja → soja;
- conversão de outras classes → soja;
- percentuais de persistência e conversão;
- emissão absoluta de CO₂;
- taxa de emissão de CO₂;
- erros-padrão e intervalos de confiança quando disponíveis;
- valores negativos existentes na fonte original;
- valores ausentes como nulos.

Os arquivos derivados anteriormente pela equipe foram utilizados
somente para auditoria e rastreabilidade, não como fonte da camada
Curated final.

In [38]:
# ============================================================
# APPENDIX B — ARQUIVO E DOCUMENTAÇÃO
# ============================================================

ARQUIVO_APPENDIX_B = (
    BRLUC_ORIGINAL_DIR
    / "Batistaetal_Appendix_B_REDAPE_v2.xlsx"
)


print(
    "Arquivo:"
)

print(
    ARQUIVO_APPENDIX_B
)


print(
    "\nExiste:"
)

print(
    ARQUIVO_APPENDIX_B.exists()
)


introducao_appendix_b = pd.read_excel(
    ARQUIVO_APPENDIX_B,
    sheet_name="Introduction",
    header=None,
    engine="openpyxl"
)


print(
    "\nDimensão da Introduction:"
)

print(
    introducao_appendix_b.shape
)


# Mostrar somente linhas com algum conteúdo
introducao_b_nao_vazia = (
    introducao_appendix_b
    .dropna(
        how="all"
    )
)


display(
    introducao_b_nao_vazia
)

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\embrapa_brluc\original_brluc_2_1\Batistaetal_Appendix_B_REDAPE_v2.xlsx

Existe:
True

Dimensão da Introduction:
(102, 23)


,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
1,NaN,Appendix B | Supplementary Data of Batista et ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,"Welcome to Appendix B, which contains the outp...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data are also available for easier visualizati...
4,NaN,"BATISTA, A. M.; GOMES, L. E. S.; PAZIANOTTO, R...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Estimated total carbon stock (SOC+Cveg) (tC.h...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Estimated soil organic carbon stock (tC.ha⁻¹),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Estimated biomass carbon stock (tC.ha⁻¹),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Standard error,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Lower limit of 95% confidence intervals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brluc.cnpma.embrapa.br
9,NaN,This workbook presents the final results for t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Upper limit of 95% confidence intervals,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Uncertainty (%),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Soil organic carbon stock in the reference co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
# ============================================================
# APPENDIX B — AMOSTRA DAS TABELAS MUNICIPAIS
# ============================================================

ABAS_B_MUNICIPAIS = [
    "C_total_Municipal_level",
    "SOC_Municipal_ level",
    "Cveg_Municipal_level"
]


for aba in ABAS_B_MUNICIPAIS:

    print("\n" + "=" * 90)
    print(aba)
    print("=" * 90)

    amostra = pd.read_excel(
        ARQUIVO_APPENDIX_B,
        sheet_name=aba,
        header=None,
        nrows=10,
        engine="openpyxl"
    )

    print(
        "Dimensão da amostra:",
        amostra.shape
    )

    display(
        amostra
    )


C_total_Municipal_level
Dimensão da amostra: (10, 227)


,0,1,2,3,4,5,6,7,8,9,...,217,218,219,220,221,222,223,224,225,226
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,State code,State name,Microregion code,Microregion name,Municipality code,Municipality name,"Natural land, average",NaN,NaN,...,"Settlements, generic",NaN,NaN,NaN,NaN,"Other land, generic",NaN,NaN,NaN,NaN
4,NaN,State code,State,Microregion code,Microregion,County code,County,Ctotal,SE,95%CI Low,...,Ctotal,SE,95%CI Low,95%CI Upp,Unc,Ctotal,SE,95%CI Low,95%CI Upp,Unc
5,NaN,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,190.750543,35.069061,128.112649,...,39.005178,3.735389,32.097475,46.538209,0.193129,48.757345,4.66854,40.043744,58.123229,0.192092
6,NaN,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,178.888852,8.154942,162.576965,...,37.416463,3.78671,30.113823,44.911863,0.200324,46.771416,4.732768,37.514824,56.030262,0.197959
7,NaN,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,134.393661,22.187879,93.851147,...,34.093156,3.728265,26.938631,41.255906,0.210093,42.617101,4.658463,33.66426,51.6052,0.210904
8,NaN,11,Rondônia,11006,Cacoal (RO),1100049,Cacoal,181.551293,8.171284,165.132867,...,39.357404,3.717662,32.219477,46.744056,0.187681,49.197372,4.643131,40.131892,58.264503,0.184301
9,NaN,11,Rondônia,11008,Colorado do Oeste (RO),1100056,Cerejeiras,132.604143,30.574487,79.84406,...,34.410705,3.764258,27.512961,42.070129,0.222588,43.014073,4.703798,34.313275,52.488613,0.220266



SOC_Municipal_ level
Dimensão da amostra: (10, 227)


,0,1,2,3,4,5,6,7,8,9,...,217,218,219,220,221,222,223,224,225,226
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,State code,State name,Microregion code,Microregion name,Municipality code,Municipality name,"Natural land, average",NaN,NaN,...,"Settlements, generic",NaN,NaN,NaN,NaN,"Other land, generic",NaN,NaN,NaN,NaN
4,NaN,State code,State,Microregion code,Microregion,County code,County,SOC,SE,95%CI Low,...,SOC,SE,95%CI Low,95%CI Upp,Unc,SOC,SE,95%CI Low,95%CI Upp,Unc
5,NaN,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,48.757345,4.66854,40.043744,...,39.005178,3.735389,32.097475,46.538209,0.193129,48.757345,4.66854,40.043744,58.123229,0.192092
6,NaN,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,46.771416,4.732768,37.514824,...,37.416463,3.78671,30.113823,44.911863,0.200324,46.771416,4.732768,37.514824,56.030262,0.197959
7,NaN,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,42.617101,4.658463,33.66426,...,34.093156,3.728265,26.938631,41.255906,0.210093,42.617101,4.658463,33.66426,51.6052,0.210904
8,NaN,11,Rondônia,11006,Cacoal (RO),1100049,Cacoal,49.197372,4.643131,40.131892,...,39.357404,3.717662,32.219477,46.744056,0.187681,49.197372,4.643131,40.131892,58.264503,0.184301
9,NaN,11,Rondônia,11008,Colorado do Oeste (RO),1100056,Cerejeiras,43.014073,4.703798,34.313275,...,34.410705,3.764258,27.512961,42.070129,0.222588,43.014073,4.703798,34.313275,52.488613,0.220266



Cveg_Municipal_level
Dimensão da amostra: (10, 227)


,0,1,2,3,4,5,6,7,8,9,...,217,218,219,220,221,222,223,224,225,226
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Return to ""Introduction"" tab",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,State code,State name,Microregion code,Microregion name,Municipality code,Municipality name,"Natural land, average",NaN,NaN,...,"Settlements, generic",NaN,NaN,NaN,NaN,"Other land, generic",NaN,NaN,NaN,NaN
4,NaN,State code,State,Microregion code,Microregion,County code,County,Cveg,SE,95%CI Low,...,Cveg,SE,95%CI Low,95%CI Upp,Unc,Cveg,SE,95%CI Low,95%CI Upp,Unc
5,NaN,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,141.993199,34.784799,79.485804,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,132.117436,6.706813,119.037865,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,91.776559,21.697503,52.891204,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,11,Rondônia,11006,Cacoal (RO),1100049,Cacoal,132.353921,6.765579,118.536205,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,11,Rondônia,11008,Colorado do Oeste (RO),1100056,Cerejeiras,89.590071,30.208654,40.100245,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
# ============================================================
# APPENDIX B — LOCALIZAÇÃO DA CLASSE SOYBEAN
# ============================================================

for aba in ABAS_B_MUNICIPAIS:

    print("\n" + "=" * 90)
    print(aba)
    print("=" * 90)

    cabecalho_bruto = pd.read_excel(
        ARQUIVO_APPENDIX_B,
        sheet_name=aba,
        header=None,
        nrows=8,
        engine="openpyxl"
    )


    ocorrencias_soja = []


    for linha in cabecalho_bruto.index:

        for coluna in cabecalho_bruto.columns:

            valor = cabecalho_bruto.loc[
                linha,
                coluna
            ]

            if (
                pd.notna(valor)
                and
                "soy" in str(valor).lower()
            ):

                ocorrencias_soja.append(
                    {
                        "linha": linha,
                        "coluna": coluna,
                        "valor": valor
                    }
                )


    print(
        "Ocorrências relacionadas a Soybean:"
    )

    display(
        pd.DataFrame(
            ocorrencias_soja
        )
    )


C_total_Municipal_level
Ocorrências relacionadas a Soybean:


""



SOC_Municipal_ level
Ocorrências relacionadas a Soybean:


""



Cveg_Municipal_level
Ocorrências relacionadas a Soybean:


""


In [41]:
# ============================================================
# MAPEAMENTO OFICIAL DA SOJA — CLASSES BRLUC EM t0 E t1
# ============================================================

linha_soja_land_use = (
    land_use_classes[
        land_use_classes[1]
        .astype("string")
        .str.strip()
        .eq("Soybean")
    ]
    .copy()
)


print(
    "Quantidade de linhas Soybean:"
)

print(
    len(linha_soja_land_use)
)


print(
    "\nConteúdo completo da linha:"
)


for coluna in linha_soja_land_use.columns:

    valor = (
        linha_soja_land_use[
            coluna
        ]
        .iloc[0]
    )

    print(
        f"Coluna {coluna}:",
        repr(valor)
    )

Quantidade de linhas Soybean:
1

Conteúdo completo da linha:
Coluna 0: np.float64(nan)
Coluna 1: 'Soybean'
Coluna 2: 'Cropland, temporary, generic, average tillage 2006, medium'
Coluna 3: nan
Coluna 4: nan
Coluna 5: 'Cropland, temporary, generic, average tillage 2017, medium'
Coluna 6: nan
Coluna 7: nan
Coluna 8: nan


In [42]:
# ============================================================
# CLASSES DE ESTOQUE DE CARBONO — APPENDIX B
# ============================================================

cabecalho_classes_b = pd.read_excel(
    ARQUIVO_APPENDIX_B,
    sheet_name="C_total_Municipal_level",
    header=None,
    nrows=5,
    engine="openpyxl"
)


classes_brluc_b = (
    cabecalho_classes_b
    .iloc[3]
    .dropna()
    .astype("string")
    .str.strip()
    .tolist()
)


# Remover campos territoriais
campos_territoriais = {
    "State code",
    "State name",
    "Microregion code",
    "Microregion name",
    "Municipality code",
    "Municipality name"
}


classes_carbono_b = [
    classe
    for classe in classes_brluc_b
    if classe not in campos_territoriais
]


print(
    "Quantidade de classes de carbono:"
)

print(
    len(
        classes_carbono_b
    )
)


print(
    "\nClasses encontradas:"
)


for indice, classe in enumerate(
    classes_carbono_b,
    start=1
):

    print(
        indice,
        "->",
        repr(classe)
    )

Quantidade de classes de carbono:
44

Classes encontradas:
1 -> 'Natural land, average'
2 -> 'Forest land, planted, eucalyptus'
3 -> 'Forest land, planted, pinus'
4 -> 'Forest land, planted, other broadleaf'
5 -> 'Forest land, planted, average species 2013'
6 -> 'Forest land, planted, average species 2022'
7 -> 'Grassland, cultivated, severely degraded, medium'
8 -> 'Grassland, cultivated, high intensity grazing, medium'
9 -> 'Grassland, cultivated, moderately degraded, medium'
10 -> 'Grassland, cultivated, non-degraded, medium'
11 -> 'Grassland, cultivated, improved, medium'
12 -> 'Grassland, cultivated, average management 2003, medium'
13 -> 'Grassland, cultivated, average management 2023, medium'
14 -> 'Cropland, temporary, generic, full tillage, low'
15 -> 'Cropland, temporary, generic, full tillage, medium'
16 -> 'Cropland, temporary, generic, reduced tillage, low'
17 -> 'Cropland, temporary, generic, reduced tillage, medium'
18 -> 'Cropland, temporary, generic, reduced tillage, h

In [43]:
# ============================================================
# CLASSES CANDIDATAS PARA REPRESENTAR SOJA
# ============================================================

termos_busca = [
    "cropland",
    "temporary",
    "tillage",
    "2006",
    "2017"
]


classes_candidatas_soja = []


for classe in classes_carbono_b:

    classe_lower = (
        str(classe)
        .lower()
    )

    if any(
        termo in classe_lower
        for termo in termos_busca
    ):

        classes_candidatas_soja.append(
            classe
        )


print(
    "Classes candidatas:"
)


for indice, classe in enumerate(
    classes_candidatas_soja,
    start=1
):

    print(
        indice,
        "->",
        repr(classe)
    )

Classes candidatas:
1 -> 'Cropland, temporary, generic, full tillage, low'
2 -> 'Cropland, temporary, generic, full tillage, medium'
3 -> 'Cropland, temporary, generic, reduced tillage, low'
4 -> 'Cropland, temporary, generic, reduced tillage, medium'
5 -> 'Cropland, temporary, generic, reduced tillage, high'
6 -> 'Cropland, temporary, generic, no till, low'
7 -> 'Cropland, temporary, generic, no till, medium'
8 -> 'Cropland, temporary, generic, no till, high'
9 -> 'Cropland, temporary, generic, average tillage 2006, medium'
10 -> 'Cropland, temporary, generic, average tillage 2017, medium'
11 -> 'Cropland, temporary, sugarcane, low'
12 -> 'Cropland, temporary, sugarcane, high'
13 -> 'Cropland, temporary, sugarcane, average 2003'
14 -> 'Cropland, temporary, sugarcane, average 2023'
15 -> 'Cropland, temporary, paddy rice'
16 -> 'Cropland, permanent, generic, full tillage, low'
17 -> 'Cropland, permanent, generic, full tillage, medium'
18 -> 'Cropland, permanent, generic, reduced tillage

In [44]:
# ============================================================
# APPENDIX B — LOCALIZAÇÃO DOS BLOCOS DA SOJA
# ============================================================

CLASSE_SOJA_T0 = (
    "Cropland, temporary, generic, average tillage 2006, medium"
)

CLASSE_SOJA_T1 = (
    "Cropland, temporary, generic, average tillage 2017, medium"
)


for aba in ABAS_B_MUNICIPAIS:

    print("\n" + "=" * 90)
    print(aba)
    print("=" * 90)

    cabecalho = pd.read_excel(
        ARQUIVO_APPENDIX_B,
        sheet_name=aba,
        header=None,
        nrows=5,
        engine="openpyxl"
    )


    indice_t0 = (
        cabecalho
        .columns[
            cabecalho
            .iloc[3]
            .eq(CLASSE_SOJA_T0)
        ]
        .tolist()
    )


    indice_t1 = (
        cabecalho
        .columns[
            cabecalho
            .iloc[3]
            .eq(CLASSE_SOJA_T1)
        ]
        .tolist()
    )


    print(
        "Início bloco t0:",
        indice_t0
    )

    print(
        "Início bloco t1:",
        indice_t1
    )


    if indice_t0:

        inicio = indice_t0[0]

        print(
            "\nSubcolunas t0:"
        )

        for coluna in range(
            inicio,
            inicio + 5
        ):

            print(
                coluna,
                "->",
                repr(
                    cabecalho.iloc[
                        4,
                        coluna
                    ]
                )
            )


    if indice_t1:

        inicio = indice_t1[0]

        print(
            "\nSubcolunas t1:"
        )

        for coluna in range(
            inicio,
            inicio + 5
        ):

            print(
                coluna,
                "->",
                repr(
                    cabecalho.iloc[
                        4,
                        coluna
                    ]
                )
            )


C_total_Municipal_level
Início bloco t0: [112]
Início bloco t1: [117]

Subcolunas t0:
112 -> 'Ctotal'
113 -> 'SE'
114 -> '95%CI Low'
115 -> '95%CI Upp'
116 -> 'Unc'

Subcolunas t1:
117 -> 'Ctotal'
118 -> 'SE'
119 -> '95%CI Low'
120 -> '95%CI Upp'
121 -> 'Unc'

SOC_Municipal_ level
Início bloco t0: [112]
Início bloco t1: [117]

Subcolunas t0:
112 -> 'SOC'
113 -> 'SE'
114 -> '95%CI Low'
115 -> '95%CI Upp'
116 -> 'Unc'

Subcolunas t1:
117 -> 'SOC'
118 -> 'SE'
119 -> '95%CI Low'
120 -> '95%CI Upp'
121 -> 'Unc'

Cveg_Municipal_level
Início bloco t0: [112]
Início bloco t1: [117]

Subcolunas t0:
112 -> 'Cveg'
113 -> 'SE'
114 -> '95%CI Low'
115 -> '95%CI Upp'
116 -> 'Unc'

Subcolunas t1:
117 -> 'Cveg'
118 -> 'SE'
119 -> '95%CI Low'
120 -> '95%CI Upp'
121 -> 'Unc'


In [45]:
# ============================================================
# FUNÇÃO — EXTRAÇÃO DAS CLASSES t0 E t1 DA SOJA
# ============================================================

def extrair_estoque_soja_appendix_b(
    aba,
    prefixo
):

    # --------------------------------------------------------
    # Ler apenas o cabeçalho para localizar os blocos
    # --------------------------------------------------------

    cabecalho = pd.read_excel(
        ARQUIVO_APPENDIX_B,
        sheet_name=aba,
        header=None,
        nrows=5,
        engine="openpyxl"
    )


    inicio_t0 = (
        cabecalho
        .columns[
            cabecalho
            .iloc[3]
            .eq(CLASSE_SOJA_T0)
        ]
        .tolist()[0]
    )


    inicio_t1 = (
        cabecalho
        .columns[
            cabecalho
            .iloc[3]
            .eq(CLASSE_SOJA_T1)
        ]
        .tolist()[0]
    )


    # --------------------------------------------------------
    # Colunas territoriais
    # --------------------------------------------------------

    colunas_identificacao = [
        1,  # State code
        2,  # State name
        3,  # Microregion code
        4,  # Microregion name
        5,  # Municipality code
        6   # Municipality name
    ]


    colunas_t0 = list(
        range(
            inicio_t0,
            inicio_t0 + 5
        )
    )


    colunas_t1 = list(
        range(
            inicio_t1,
            inicio_t1 + 5
        )
    )


    colunas_leitura = (
        colunas_identificacao
        +
        colunas_t0
        +
        colunas_t1
    )


    # --------------------------------------------------------
    # Ler somente as colunas necessárias
    # Dados começam na linha 5
    # --------------------------------------------------------

    base = pd.read_excel(
        ARQUIVO_APPENDIX_B,
        sheet_name=aba,
        header=None,
        skiprows=5,
        usecols=colunas_leitura,
        engine="openpyxl"
    )


    base.columns = [
        "codigo_uf",
        "estado",
        "codigo_microrregiao",
        "microrregiao",
        "codigo_ibge",
        "municipio",

        f"{prefixo}_t0_t_c_ha",
        f"{prefixo}_t0_se",
        f"{prefixo}_t0_ic95_inf",
        f"{prefixo}_t0_ic95_sup",
        f"{prefixo}_t0_incerteza",

        f"{prefixo}_t1_t_c_ha",
        f"{prefixo}_t1_se",
        f"{prefixo}_t1_ic95_inf",
        f"{prefixo}_t1_ic95_sup",
        f"{prefixo}_t1_incerteza"
    ]


    # --------------------------------------------------------
    # Padronizar código IBGE
    # --------------------------------------------------------

    base[
        "codigo_ibge"
    ] = (
        pd.to_numeric(
            base[
                "codigo_ibge"
            ],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
        .str.zfill(7)
    )


    return base

In [46]:
# ============================================================
# EXTRAÇÃO — CTOTAL, SOC E CVEG
# ============================================================

ctotal_soja_b = extrair_estoque_soja_appendix_b(
    aba="C_total_Municipal_level",
    prefixo="ctotal"
)


soc_soja_b = extrair_estoque_soja_appendix_b(
    aba="SOC_Municipal_ level",
    prefixo="soc"
)


cveg_soja_b = extrair_estoque_soja_appendix_b(
    aba="Cveg_Municipal_level",
    prefixo="cveg"
)


for nome, base in {
    "Ctotal": ctotal_soja_b,
    "SOC": soc_soja_b,
    "Cveg": cveg_soja_b
}.items():

    print("\n" + "=" * 80)

    print(
        nome
    )

    print(
        "Dimensão:",
        base.shape
    )

    print(
        "Códigos distintos:",
        base[
            "codigo_ibge"
        ]
        .nunique()
    )

    print(
        "Duplicatas:",
        base
        .duplicated(
            subset=[
                "codigo_ibge"
            ]
        )
        .sum()
    )

    display(
        base.head(3)
    )


Ctotal
Dimensão: (5569, 16)
Códigos distintos: 5569
Duplicatas: 0


,codigo_uf,estado,codigo_microrregiao,microrregiao,codigo_ibge,municipio,ctotal_t0_t_c_ha,ctotal_t0_se,ctotal_t0_ic95_inf,ctotal_t0_ic95_sup,ctotal_t0_incerteza,ctotal_t1_t_c_ha,ctotal_t1_se,ctotal_t1_ic95_inf,ctotal_t1_ic95_sup,ctotal_t1_incerteza
0,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,45.731350,4.916691,36.180090,55.231262,0.208856,46.237999,4.970651,36.827520,56.054846,0.212311
1,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,44.169479,4.886836,34.931867,53.919700,0.220746,45.391372,5.037475,35.790330,55.365195,0.219729
2,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,41.627819,4.889697,32.344965,51.163868,0.229079,42.500954,5.014898,33.014297,52.271642,0.229893



SOC
Dimensão: (5569, 16)
Códigos distintos: 5569
Duplicatas: 0


,codigo_uf,estado,codigo_microrregiao,microrregiao,codigo_ibge,municipio,soc_t0_t_c_ha,soc_t0_se,soc_t0_ic95_inf,soc_t0_ic95_sup,soc_t0_incerteza,soc_t1_t_c_ha,soc_t1_se,soc_t1_ic95_inf,soc_t1_ic95_sup,soc_t1_incerteza
0,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,41.025475,4.586641,32.307318,49.989542,0.218500,41.532124,4.644106,32.903569,50.816968,0.223558
1,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,39.463604,4.561750,30.727862,48.325712,0.224564,40.685497,4.722266,31.653346,49.794798,0.223896
2,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,36.921944,4.556438,28.272195,45.689166,0.237453,37.795079,4.689948,29.131744,47.040445,0.244618



Cveg
Dimensão: (5569, 16)
Códigos distintos: 5569
Duplicatas: 0


,codigo_uf,estado,codigo_microrregiao,microrregiao,codigo_ibge,municipio,cveg_t0_t_c_ha,cveg_t0_se,cveg_t0_ic95_inf,cveg_t0_ic95_sup,cveg_t0_incerteza,cveg_t1_t_c_ha,cveg_t1_se,cveg_t1_ic95_inf,cveg_t1_ic95_sup,cveg_t1_incerteza
0,11,Rondônia,11006,Cacoal (RO),1100015,Alta Floresta D'Oeste,4.705875,1.781324,1.832727,8.266892,0.756717,4.705875,1.781324,1.832727,8.266892,0.756717
1,11,Rondônia,11003,Ariquemes (RO),1100023,Ariquemes,4.705875,1.781324,1.832727,8.266892,0.756717,4.705875,1.781324,1.832727,8.266892,0.756717
2,11,Rondônia,11008,Colorado do Oeste (RO),1100031,Cabixi,4.705875,1.781324,1.832727,8.266892,0.756717,4.705875,1.781324,1.832727,8.266892,0.756717


In [47]:
# ============================================================
# APPENDIX B — VALIDAÇÃO CTOTAL = SOC + CVEG
# ============================================================

estoques_b_validacao = (
    ctotal_soja_b[
        [
            "codigo_ibge",
            "municipio",
            "ctotal_t0_t_c_ha",
            "ctotal_t1_t_c_ha"
        ]
    ]
    .merge(
        soc_soja_b[
            [
                "codigo_ibge",
                "soc_t0_t_c_ha",
                "soc_t1_t_c_ha"
            ]
        ],
        on="codigo_ibge",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        cveg_soja_b[
            [
                "codigo_ibge",
                "cveg_t0_t_c_ha",
                "cveg_t1_t_c_ha"
            ]
        ],
        on="codigo_ibge",
        how="inner",
        validate="one_to_one"
    )
)


estoques_b_validacao[
    "ctotal_t0_recalculado"
] = (
    estoques_b_validacao[
        "soc_t0_t_c_ha"
    ]
    +
    estoques_b_validacao[
        "cveg_t0_t_c_ha"
    ]
)


estoques_b_validacao[
    "ctotal_t1_recalculado"
] = (
    estoques_b_validacao[
        "soc_t1_t_c_ha"
    ]
    +
    estoques_b_validacao[
        "cveg_t1_t_c_ha"
    ]
)


estoques_b_validacao[
    "dif_ctotal_t0"
] = (
    estoques_b_validacao[
        "ctotal_t0_t_c_ha"
    ]
    -
    estoques_b_validacao[
        "ctotal_t0_recalculado"
    ]
).abs()


estoques_b_validacao[
    "dif_ctotal_t1"
] = (
    estoques_b_validacao[
        "ctotal_t1_t_c_ha"
    ]
    -
    estoques_b_validacao[
        "ctotal_t1_recalculado"
    ]
).abs()


print("=" * 70)
print("VALIDAÇÃO CTOTAL = SOC + CVEG")
print("=" * 70)


print(
    "\nRegistros:"
)

print(
    estoques_b_validacao.shape
)


print(
    "\nMaior diferença t0:"
)

print(
    estoques_b_validacao[
        "dif_ctotal_t0"
    ]
    .max()
)


print(
    "\nFalhas t0 > 0.000001:"
)

print(
    (
        estoques_b_validacao[
            "dif_ctotal_t0"
        ]
        > 0.000001
    )
    .sum()
)


print(
    "\nMaior diferença t1:"
)

print(
    estoques_b_validacao[
        "dif_ctotal_t1"
    ]
    .max()
)


print(
    "\nFalhas t1 > 0.000001:"
)

print(
    (
        estoques_b_validacao[
            "dif_ctotal_t1"
        ]
        > 0.000001
    )
    .sum()
)


display(
    estoques_b_validacao[
        [
            "codigo_ibge",
            "municipio",

            "soc_t0_t_c_ha",
            "cveg_t0_t_c_ha",
            "ctotal_t0_t_c_ha",

            "soc_t1_t_c_ha",
            "cveg_t1_t_c_ha",
            "ctotal_t1_t_c_ha"
        ]
    ]
    .head(10)
)

VALIDAÇÃO CTOTAL = SOC + CVEG

Registros:
(5569, 12)

Maior diferença t0:
8.526512829121202e-14

Falhas t0 > 0.000001:
0

Maior diferença t1:
8.526512829121202e-14

Falhas t1 > 0.000001:
0


,codigo_ibge,municipio,soc_t0_t_c_ha,cveg_t0_t_c_ha,ctotal_t0_t_c_ha,soc_t1_t_c_ha,cveg_t1_t_c_ha,ctotal_t1_t_c_ha
0,1100015,Alta Floresta D'Oeste,41.025475,4.705875,45.731350,41.532124,4.705875,46.237999
1,1100023,Ariquemes,39.463604,4.705875,44.169479,40.685497,4.705875,45.391372
2,1100031,Cabixi,36.921944,4.705875,41.627819,37.795079,4.705875,42.500954
3,1100049,Cacoal,41.173865,4.705875,45.879740,41.923327,4.705875,46.629202
4,1100056,Cerejeiras,36.966661,4.705875,41.672536,39.205373,4.705875,43.911248
5,1100064,Colorado do Oeste,30.462321,4.705875,35.168196,32.753960,4.705875,37.459835
6,1100072,Corumbiara,34.417157,4.705875,39.123032,37.129110,4.705875,41.834985
7,1100080,Costa Marques,41.366829,4.705875,46.072704,43.505883,4.705875,48.211758
8,1100098,Espigão D'Oeste,41.258042,4.705875,45.963917,43.673874,4.705875,48.379749
9,1100106,Guajará-Mirim,42.420476,4.705875,47.126351,42.491441,4.705875,47.197316


In [48]:
# ============================================================
# APPENDIX B — RECORTE CENTRO-OESTE + SUL
# ============================================================

CODIGOS_PROJETO_BRLUC = set(
    brluc_curated[
        "codigo_ibge"
    ]
    .astype("string")
)


ctotal_soja_projeto_b = (
    ctotal_soja_b[
        ctotal_soja_b[
            "codigo_ibge"
        ]
        .isin(
            CODIGOS_PROJETO_BRLUC
        )
    ]
    .copy()
)


soc_soja_projeto_b = (
    soc_soja_b[
        soc_soja_b[
            "codigo_ibge"
        ]
        .isin(
            CODIGOS_PROJETO_BRLUC
        )
    ]
    .copy()
)


cveg_soja_projeto_b = (
    cveg_soja_b[
        cveg_soja_b[
            "codigo_ibge"
        ]
        .isin(
            CODIGOS_PROJETO_BRLUC
        )
    ]
    .copy()
)


for nome, base in {
    "Ctotal": ctotal_soja_projeto_b,
    "SOC": soc_soja_projeto_b,
    "Cveg": cveg_soja_projeto_b
}.items():

    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)

    print(
        "Dimensão:",
        base.shape
    )

    print(
        "Municípios distintos:",
        base[
            "codigo_ibge"
        ]
        .nunique()
    )

    print(
        "Duplicatas:",
        base
        .duplicated(
            subset=[
                "codigo_ibge"
            ]
        )
        .sum()
    )

    print(
        "Mesmo universo do Appendix C:",
        set(
            base[
                "codigo_ibge"
            ]
        )
        ==
        CODIGOS_PROJETO_BRLUC
    )


Ctotal
Dimensão: (1658, 16)
Municípios distintos: 1658
Duplicatas: 0
Mesmo universo do Appendix C: True

SOC
Dimensão: (1658, 16)
Municípios distintos: 1658
Duplicatas: 0
Mesmo universo do Appendix C: True

Cveg
Dimensão: (1658, 16)
Municípios distintos: 1658
Duplicatas: 0
Mesmo universo do Appendix C: True


In [49]:
# ============================================================
# APPENDIX B — AUDITORIA DOS ESTOQUES DO PROJETO
# ============================================================

estoques_soja_projeto_b = (
    ctotal_soja_projeto_b[
        [
            "codigo_ibge",
            "municipio",
            "estado",

            "ctotal_t0_t_c_ha",
            "ctotal_t0_se",
            "ctotal_t0_ic95_inf",
            "ctotal_t0_ic95_sup",
            "ctotal_t0_incerteza",

            "ctotal_t1_t_c_ha",
            "ctotal_t1_se",
            "ctotal_t1_ic95_inf",
            "ctotal_t1_ic95_sup",
            "ctotal_t1_incerteza"
        ]
    ]
    .merge(
        soc_soja_projeto_b[
            [
                "codigo_ibge",

                "soc_t0_t_c_ha",
                "soc_t0_se",
                "soc_t0_ic95_inf",
                "soc_t0_ic95_sup",
                "soc_t0_incerteza",

                "soc_t1_t_c_ha",
                "soc_t1_se",
                "soc_t1_ic95_inf",
                "soc_t1_ic95_sup",
                "soc_t1_incerteza"
            ]
        ],
        on="codigo_ibge",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        cveg_soja_projeto_b[
            [
                "codigo_ibge",

                "cveg_t0_t_c_ha",
                "cveg_t0_se",
                "cveg_t0_ic95_inf",
                "cveg_t0_ic95_sup",
                "cveg_t0_incerteza",

                "cveg_t1_t_c_ha",
                "cveg_t1_se",
                "cveg_t1_ic95_inf",
                "cveg_t1_ic95_sup",
                "cveg_t1_incerteza"
            ]
        ],
        on="codigo_ibge",
        how="inner",
        validate="one_to_one"
    )
)


COLUNAS_ESTOQUE_PRINCIPAIS = [
    "ctotal_t0_t_c_ha",
    "ctotal_t1_t_c_ha",
    "soc_t0_t_c_ha",
    "soc_t1_t_c_ha",
    "cveg_t0_t_c_ha",
    "cveg_t1_t_c_ha"
]


print(
    "Dimensão integrada:"
)

print(
    estoques_soja_projeto_b.shape
)


print(
    "\nResumo estatístico:"
)

display(
    estoques_soja_projeto_b[
        COLUNAS_ESTOQUE_PRINCIPAIS
    ]
    .describe()
    .T
)


print(
    "\nNulos:"
)

display(
    estoques_soja_projeto_b[
        COLUNAS_ESTOQUE_PRINCIPAIS
    ]
    .isna()
    .sum()
)


print(
    "\nNegativos:"
)

for coluna in COLUNAS_ESTOQUE_PRINCIPAIS:

    print(
        coluna,
        "->",
        (
            estoques_soja_projeto_b[
                coluna
            ]
            < 0
        )
        .sum()
    )

Dimensão integrada:
(1658, 33)

Resumo estatístico:


,count,mean,std,min,25%,50%,75%,max
ctotal_t0_t_c_ha,1658.0,50.012745,1.814802e+01,20.766784,38.794348,42.939029,56.051160,98.097215
ctotal_t1_t_c_ha,1658.0,51.196322,1.884688e+01,20.755812,39.458743,43.558657,57.432523,98.478138
soc_t0_t_c_ha,1658.0,45.306870,1.814802e+01,16.060909,34.088473,38.233154,51.345285,93.391340
soc_t1_t_c_ha,1658.0,46.490447,1.884688e+01,16.049937,34.752868,38.852782,52.726648,93.772263
cveg_t0_t_c_ha,1658.0,4.705875,1.003944e-13,4.705875,4.705875,4.705875,4.705875,4.705875
cveg_t1_t_c_ha,1658.0,4.705875,1.003944e-13,4.705875,4.705875,4.705875,4.705875,4.705875



Nulos:


ctotal_t0_t_c_ha    0
ctotal_t1_t_c_ha    0
soc_t0_t_c_ha       0
soc_t1_t_c_ha       0
cveg_t0_t_c_ha      0
cveg_t1_t_c_ha      0
dtype: int64


Negativos:
ctotal_t0_t_c_ha -> 0
ctotal_t1_t_c_ha -> 0
soc_t0_t_c_ha -> 0
soc_t1_t_c_ha -> 0
cveg_t0_t_c_ha -> 0
cveg_t1_t_c_ha -> 0


In [50]:
# ============================================================
# APPENDIX B — COMPARAÇÃO ENTRE AS CLASSES t0 E t1
# ============================================================

estoques_soja_projeto_b[
    "delta_soc_classe_t1_t0_t_c_ha"
] = (
    estoques_soja_projeto_b[
        "soc_t1_t_c_ha"
    ]
    -
    estoques_soja_projeto_b[
        "soc_t0_t_c_ha"
    ]
)


estoques_soja_projeto_b[
    "delta_cveg_classe_t1_t0_t_c_ha"
] = (
    estoques_soja_projeto_b[
        "cveg_t1_t_c_ha"
    ]
    -
    estoques_soja_projeto_b[
        "cveg_t0_t_c_ha"
    ]
)


estoques_soja_projeto_b[
    "delta_ctotal_classe_t1_t0_t_c_ha"
] = (
    estoques_soja_projeto_b[
        "ctotal_t1_t_c_ha"
    ]
    -
    estoques_soja_projeto_b[
        "ctotal_t0_t_c_ha"
    ]
)


print("=" * 70)
print("COMPARAÇÃO DAS CLASSES ASSOCIADAS À SOJA")
print("=" * 70)


print(
    "\nCveg t0 e t1 exatamente iguais:"
)

print(
    np.allclose(
        estoques_soja_projeto_b[
            "cveg_t0_t_c_ha"
        ],
        estoques_soja_projeto_b[
            "cveg_t1_t_c_ha"
        ],
        equal_nan=True
    )
)


print(
    "\nMaior diferença absoluta de Cveg:"
)

print(
    (
        estoques_soja_projeto_b[
            "cveg_t1_t_c_ha"
        ]
        -
        estoques_soja_projeto_b[
            "cveg_t0_t_c_ha"
        ]
    )
    .abs()
    .max()
)


print(
    "\nDelta Ctotal = Delta SOC:"
)

print(
    np.allclose(
        estoques_soja_projeto_b[
            "delta_ctotal_classe_t1_t0_t_c_ha"
        ],
        estoques_soja_projeto_b[
            "delta_soc_classe_t1_t0_t_c_ha"
        ],
        equal_nan=True
    )
)


print(
    "\nResumo dos deltas:"
)

display(
    estoques_soja_projeto_b[
        [
            "delta_soc_classe_t1_t0_t_c_ha",
            "delta_cveg_classe_t1_t0_t_c_ha",
            "delta_ctotal_classe_t1_t0_t_c_ha"
        ]
    ]
    .describe()
    .T
)


print(
    "\nDelta SOC positivo:"
)

print(
    (
        estoques_soja_projeto_b[
            "delta_soc_classe_t1_t0_t_c_ha"
        ]
        > 0
    )
    .sum()
)


print(
    "\nDelta SOC zero:"
)

print(
    np.isclose(
        estoques_soja_projeto_b[
            "delta_soc_classe_t1_t0_t_c_ha"
        ],
        0
    )
    .sum()
)


print(
    "\nDelta SOC negativo:"
)

print(
    (
        estoques_soja_projeto_b[
            "delta_soc_classe_t1_t0_t_c_ha"
        ]
        < 0
    )
    .sum()
)


display(
    estoques_soja_projeto_b[
        [
            "codigo_ibge",
            "municipio",

            "soc_t0_t_c_ha",
            "soc_t1_t_c_ha",
            "delta_soc_classe_t1_t0_t_c_ha",

            "cveg_t0_t_c_ha",
            "cveg_t1_t_c_ha",

            "ctotal_t0_t_c_ha",
            "ctotal_t1_t_c_ha",
            "delta_ctotal_classe_t1_t0_t_c_ha"
        ]
    ]
    .head(10)
)

COMPARAÇÃO DAS CLASSES ASSOCIADAS À SOJA

Cveg t0 e t1 exatamente iguais:
True

Maior diferença absoluta de Cveg:
0.0

Delta Ctotal = Delta SOC:
True

Resumo dos deltas:


,count,mean,std,min,25%,50%,75%,max
delta_soc_classe_t1_t0_t_c_ha,1658.0,1.183577,1.238191,-1.932864,0.328738,0.892407,1.725661,6.211851
delta_cveg_classe_t1_t0_t_c_ha,1658.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
delta_ctotal_classe_t1_t0_t_c_ha,1658.0,1.183577,1.238191,-1.932864,0.328738,0.892407,1.725661,6.211851



Delta SOC positivo:
1482

Delta SOC zero:
20

Delta SOC negativo:
156


,codigo_ibge,municipio,soc_t0_t_c_ha,soc_t1_t_c_ha,delta_soc_classe_t1_t0_t_c_ha,cveg_t0_t_c_ha,cveg_t1_t_c_ha,ctotal_t0_t_c_ha,ctotal_t1_t_c_ha,delta_ctotal_classe_t1_t0_t_c_ha
0,4100103,Abatiá,27.281039,28.213927,0.932888,4.705875,4.705875,31.986914,32.919802,0.932888
1,4100202,Adrianópolis,37.900967,40.084794,2.183826,4.705875,4.705875,42.606842,44.790669,2.183826
2,4100301,Agudos do Sul,39.951615,40.874459,0.922843,4.705875,4.705875,44.657490,45.580334,0.922843
3,4100400,Almirante Tamandaré,71.114890,73.762099,2.647209,4.705875,4.705875,75.820765,78.467974,2.647209
4,4100459,Altamira do Paraná,54.017745,55.254553,1.236808,4.705875,4.705875,58.723620,59.960428,1.236808
5,4100509,Altônia,34.507703,35.412039,0.904336,4.705875,4.705875,39.213578,40.117914,0.904336
6,4100608,Alto Paraná,37.527741,37.426438,-0.101303,4.705875,4.705875,42.233616,42.132313,-0.101303
7,4100707,Alto Piquiri,42.439800,44.123370,1.683570,4.705875,4.705875,47.145675,48.829245,1.683570
8,4100806,Alvorada do Sul,39.427967,40.027123,0.599156,4.705875,4.705875,44.133842,44.732998,0.599156
9,4100905,Amaporã,37.585572,38.066157,0.480585,4.705875,4.705875,42.291447,42.772032,0.480585


In [51]:
# ============================================================
# APPENDIX B — AUDITORIA DAS INCERTEZAS
# ============================================================

COLUNAS_INcerteza_B = [
    # Ctotal
    "ctotal_t0_se",
    "ctotal_t0_ic95_inf",
    "ctotal_t0_ic95_sup",
    "ctotal_t0_incerteza",

    "ctotal_t1_se",
    "ctotal_t1_ic95_inf",
    "ctotal_t1_ic95_sup",
    "ctotal_t1_incerteza",

    # SOC
    "soc_t0_se",
    "soc_t0_ic95_inf",
    "soc_t0_ic95_sup",
    "soc_t0_incerteza",

    "soc_t1_se",
    "soc_t1_ic95_inf",
    "soc_t1_ic95_sup",
    "soc_t1_incerteza",

    # Cveg
    "cveg_t0_se",
    "cveg_t0_ic95_inf",
    "cveg_t0_ic95_sup",
    "cveg_t0_incerteza",

    "cveg_t1_se",
    "cveg_t1_ic95_inf",
    "cveg_t1_ic95_sup",
    "cveg_t1_incerteza"
]


print("=" * 70)
print("AUDITORIA DAS INCERTEZAS — APPENDIX B")
print("=" * 70)


print(
    "\nNulos:"
)

display(
    estoques_soja_projeto_b[
        COLUNAS_INcerteza_B
    ]
    .isna()
    .sum()
)


print(
    "\nValores negativos em SE:"
)

for coluna in [
    "ctotal_t0_se",
    "ctotal_t1_se",
    "soc_t0_se",
    "soc_t1_se",
    "cveg_t0_se",
    "cveg_t1_se"
]:

    print(
        coluna,
        "->",
        (
            estoques_soja_projeto_b[
                coluna
            ]
            < 0
        )
        .sum()
    )


print(
    "\nIC inferior maior que IC superior:"
)

pares_ic = [
    ("ctotal_t0_ic95_inf", "ctotal_t0_ic95_sup"),
    ("ctotal_t1_ic95_inf", "ctotal_t1_ic95_sup"),

    ("soc_t0_ic95_inf", "soc_t0_ic95_sup"),
    ("soc_t1_ic95_inf", "soc_t1_ic95_sup"),

    ("cveg_t0_ic95_inf", "cveg_t0_ic95_sup"),
    ("cveg_t1_ic95_inf", "cveg_t1_ic95_sup")
]


for inferior, superior in pares_ic:

    print(
        inferior,
        "->",
        (
            estoques_soja_projeto_b[
                inferior
            ]
            >
            estoques_soja_projeto_b[
                superior
            ]
        )
        .sum()
    )

AUDITORIA DAS INCERTEZAS — APPENDIX B

Nulos:


ctotal_t0_se           0
ctotal_t0_ic95_inf     0
ctotal_t0_ic95_sup     0
ctotal_t0_incerteza    0
ctotal_t1_se           0
ctotal_t1_ic95_inf     0
ctotal_t1_ic95_sup     0
ctotal_t1_incerteza    0
soc_t0_se              0
soc_t0_ic95_inf        0
soc_t0_ic95_sup        0
soc_t0_incerteza       0
soc_t1_se              0
soc_t1_ic95_inf        0
soc_t1_ic95_sup        0
soc_t1_incerteza       0
cveg_t0_se             0
cveg_t0_ic95_inf       0
cveg_t0_ic95_sup       0
cveg_t0_incerteza      0
cveg_t1_se             0
cveg_t1_ic95_inf       0
cveg_t1_ic95_sup       0
cveg_t1_incerteza      0
dtype: int64


Valores negativos em SE:
ctotal_t0_se -> 0
ctotal_t1_se -> 0
soc_t0_se -> 0
soc_t1_se -> 0
cveg_t0_se -> 0
cveg_t1_se -> 0

IC inferior maior que IC superior:
ctotal_t0_ic95_inf -> 0
ctotal_t1_ic95_inf -> 0
soc_t0_ic95_inf -> 0
soc_t1_ic95_inf -> 0
cveg_t0_ic95_inf -> 0
cveg_t1_ic95_inf -> 0


In [52]:
# ============================================================
# INDICADORES DERIVADOS DAS CLASSES BRLUC
# ============================================================

estoques_soja_projeto_b[
    "delta_soc_classes_brluc_t1_t0_t_c_ha"
] = (
    estoques_soja_projeto_b[
        "soc_t1_t_c_ha"
    ]
    -
    estoques_soja_projeto_b[
        "soc_t0_t_c_ha"
    ]
)


estoques_soja_projeto_b[
    "delta_cveg_classes_brluc_t1_t0_t_c_ha"
] = (
    estoques_soja_projeto_b[
        "cveg_t1_t_c_ha"
    ]
    -
    estoques_soja_projeto_b[
        "cveg_t0_t_c_ha"
    ]
)


estoques_soja_projeto_b[
    "delta_ctotal_classes_brluc_t1_t0_t_c_ha"
] = (
    estoques_soja_projeto_b[
        "ctotal_t1_t_c_ha"
    ]
    -
    estoques_soja_projeto_b[
        "ctotal_t0_t_c_ha"
    ]
)


print("=" * 70)
print("INDICADORES DERIVADOS — CLASSES BRLUC")
print("=" * 70)


for coluna in [
    "delta_soc_classes_brluc_t1_t0_t_c_ha",
    "delta_cveg_classes_brluc_t1_t0_t_c_ha",
    "delta_ctotal_classes_brluc_t1_t0_t_c_ha"
]:

    print(
        "\n",
        coluna
    )

    print(
        "Nulos:",
        estoques_soja_projeto_b[
            coluna
        ]
        .isna()
        .sum()
    )

    print(
        "Positivos:",
        (
            estoques_soja_projeto_b[
                coluna
            ]
            > 0
        )
        .sum()
    )

    print(
        "Zero:",
        np.isclose(
            estoques_soja_projeto_b[
                coluna
            ],
            0
        )
        .sum()
    )

    print(
        "Negativos:",
        (
            estoques_soja_projeto_b[
                coluna
            ]
            < 0
        )
        .sum()
    )

INDICADORES DERIVADOS — CLASSES BRLUC

 delta_soc_classes_brluc_t1_t0_t_c_ha
Nulos: 0
Positivos: 1482
Zero: 20
Negativos: 156

 delta_cveg_classes_brluc_t1_t0_t_c_ha
Nulos: 0
Positivos: 0
Zero: 1658
Negativos: 0

 delta_ctotal_classes_brluc_t1_t0_t_c_ha
Nulos: 0
Positivos: 1482
Zero: 20
Negativos: 156


In [53]:
# ============================================================
# APPENDIX B — CAMADA DE ESTOQUES PARA O PROJETO
# ============================================================

brluc_estoques_curated = (
    estoques_soja_projeto_b
    .copy()
)


# ------------------------------------------------------------
# UF e região a partir do código IBGE
# ------------------------------------------------------------

MAPA_CODIGO_UF_BRLUC = {
    "53": "DF",
    "52": "GO",
    "50": "MS",
    "51": "MT",
    "41": "PR",
    "43": "RS",
    "42": "SC"
}


brluc_estoques_curated[
    "uf"
] = (
    brluc_estoques_curated[
        "codigo_ibge"
    ]
    .str[:2]
    .map(
        MAPA_CODIGO_UF_BRLUC
    )
)


brluc_estoques_curated[
    "regiao"
] = (
    brluc_estoques_curated[
        "uf"
    ]
    .map(
        MAPA_REGIAO
    )
)


# ------------------------------------------------------------
# Metadados metodológicos
# ------------------------------------------------------------

brluc_estoques_curated[
    "cultura_representada"
] = "Soja"


brluc_estoques_curated[
    "periodo_inicio_brluc"
] = 2000


brluc_estoques_curated[
    "periodo_fim_brluc"
] = 2019


brluc_estoques_curated[
    "classe_brluc_t0"
] = CLASSE_SOJA_T0


brluc_estoques_curated[
    "classe_brluc_t1"
] = CLASSE_SOJA_T1


brluc_estoques_curated[
    "fonte"
] = "Embrapa BRLUC"


brluc_estoques_curated[
    "versao_fonte"
] = "BRLUC 2.1"


print(
    "Dimensão:"
)

print(
    brluc_estoques_curated.shape
)


print(
    "\nMunicípios:"
)

print(
    brluc_estoques_curated[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    brluc_estoques_curated
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nDistribuição por UF:"
)

display(
    brluc_estoques_curated[
        "uf"
    ]
    .value_counts()
    .sort_index()
)


display(
    brluc_estoques_curated.head()
)

Dimensão:
(1658, 48)

Municípios:
1658

Duplicatas:
0

Distribuição por UF:


uf
DF      1
GO    246
MS     79
MT    141
PR    399
RS    497
SC    295
Name: count, dtype: int64

,codigo_ibge,municipio,estado,ctotal_t0_t_c_ha,ctotal_t0_se,ctotal_t0_ic95_inf,ctotal_t0_ic95_sup,ctotal_t0_incerteza,ctotal_t1_t_c_ha,ctotal_t1_se,...,delta_ctotal_classes_brluc_t1_t0_t_c_ha,uf,regiao,cultura_representada,periodo_inicio_brluc,periodo_fim_brluc,classe_brluc_t0,classe_brluc_t1,fonte,versao_fonte
0,4100103,Abatiá,Paraná,31.986914,5.996241,21.543618,44.477164,0.390480,32.919802,6.205433,...,0.932888,PR,Sul,Soja,2000,2019,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",Embrapa BRLUC,BRLUC 2.1
1,4100202,Adrianópolis,Paraná,42.606842,5.959154,31.404718,54.445569,0.277860,44.790669,6.310514,...,2.183826,PR,Sul,Soja,2000,2019,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",Embrapa BRLUC,BRLUC 2.1
2,4100301,Agudos do Sul,Paraná,44.657490,5.831785,33.545015,55.916646,0.252122,45.580334,5.959076,...,0.922843,PR,Sul,Soja,2000,2019,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",Embrapa BRLUC,BRLUC 2.1
3,4100400,Almirante Tamandaré,Paraná,75.820765,7.609789,61.054837,90.463825,0.194748,78.467974,7.921938,...,2.647209,PR,Sul,Soja,2000,2019,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",Embrapa BRLUC,BRLUC 2.1
4,4100459,Altamira do Paraná,Paraná,58.723620,6.547916,46.533862,71.900557,0.224389,59.960428,6.729316,...,1.236808,PR,Sul,Soja,2000,2019,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",Embrapa BRLUC,BRLUC 2.1


In [54]:
# ============================================================
# APPENDIX B — INTERPRETAÇÃO DA COLUNA UNC
# ============================================================

testes_incerteza = [
    (
        "ctotal_t0",
        "ctotal_t0_t_c_ha",
        "ctotal_t0_ic95_sup",
        "ctotal_t0_incerteza"
    ),
    (
        "ctotal_t1",
        "ctotal_t1_t_c_ha",
        "ctotal_t1_ic95_sup",
        "ctotal_t1_incerteza"
    ),
    (
        "soc_t0",
        "soc_t0_t_c_ha",
        "soc_t0_ic95_sup",
        "soc_t0_incerteza"
    ),
    (
        "soc_t1",
        "soc_t1_t_c_ha",
        "soc_t1_ic95_sup",
        "soc_t1_incerteza"
    ),
    (
        "cveg_t0",
        "cveg_t0_t_c_ha",
        "cveg_t0_ic95_sup",
        "cveg_t0_incerteza"
    ),
    (
        "cveg_t1",
        "cveg_t1_t_c_ha",
        "cveg_t1_ic95_sup",
        "cveg_t1_incerteza"
    )
]


print("=" * 70)
print("VALIDAÇÃO DA COLUNA UNC")
print("=" * 70)


for nome, estimativa, ic_sup, incerteza in testes_incerteza:

    incerteza_recalculada = (
        (
            estoques_soja_projeto_b[
                ic_sup
            ]
            -
            estoques_soja_projeto_b[
                estimativa
            ]
        )
        /
        estoques_soja_projeto_b[
            estimativa
        ]
        .replace(0, np.nan)
    )


    diferenca = (
        incerteza_recalculada
        -
        estoques_soja_projeto_b[
            incerteza
        ]
    ).abs()


    print(
        f"\n{nome}"
    )

    print(
        "Maior diferença:",
        diferenca.max()
    )

    print(
        "Falhas > 0.000001:",
        (
            diferenca
            > 0.000001
        )
        .sum()
    )

VALIDAÇÃO DA COLUNA UNC

ctotal_t0
Maior diferença: 0.02211117570046925
Falhas > 0.000001: 182

ctotal_t1
Maior diferença: 0.023810866863461455
Falhas > 0.000001: 226

soc_t0
Maior diferença: 0.026866243331595374
Falhas > 0.000001: 204

soc_t1
Maior diferença: 0.02443574373943802
Falhas > 0.000001: 182

cveg_t0
Maior diferença: 1.4432899320127035e-15
Falhas > 0.000001: 0

cveg_t1
Maior diferença: 1.4432899320127035e-15
Falhas > 0.000001: 0


In [55]:
# ============================================================
# APPENDIX B — SELEÇÃO FINAL PARA INTEGRAÇÃO
# ============================================================

COLUNAS_ESTOQUES_FINAL = [
    "codigo_ibge",

    # --------------------------------------------------------
    # Classes metodológicas
    # --------------------------------------------------------
    "classe_brluc_t0",
    "classe_brluc_t1",

    # --------------------------------------------------------
    # CTOTAL
    # --------------------------------------------------------
    "ctotal_t0_t_c_ha",
    "ctotal_t0_se",
    "ctotal_t0_ic95_inf",
    "ctotal_t0_ic95_sup",
    "ctotal_t0_incerteza",

    "ctotal_t1_t_c_ha",
    "ctotal_t1_se",
    "ctotal_t1_ic95_inf",
    "ctotal_t1_ic95_sup",
    "ctotal_t1_incerteza",

    # --------------------------------------------------------
    # SOC
    # --------------------------------------------------------
    "soc_t0_t_c_ha",
    "soc_t0_se",
    "soc_t0_ic95_inf",
    "soc_t0_ic95_sup",
    "soc_t0_incerteza",

    "soc_t1_t_c_ha",
    "soc_t1_se",
    "soc_t1_ic95_inf",
    "soc_t1_ic95_sup",
    "soc_t1_incerteza",

    # --------------------------------------------------------
    # CVEG
    # --------------------------------------------------------
    "cveg_t0_t_c_ha",
    "cveg_t0_se",
    "cveg_t0_ic95_inf",
    "cveg_t0_ic95_sup",
    "cveg_t0_incerteza",

    "cveg_t1_t_c_ha",
    "cveg_t1_se",
    "cveg_t1_ic95_inf",
    "cveg_t1_ic95_sup",
    "cveg_t1_incerteza",

    # --------------------------------------------------------
    # Deltas metodológicos
    # --------------------------------------------------------
    "delta_soc_classes_brluc_t1_t0_t_c_ha",
    "delta_cveg_classes_brluc_t1_t0_t_c_ha",
    "delta_ctotal_classes_brluc_t1_t0_t_c_ha"
]


brluc_estoques_final = (
    brluc_estoques_curated[
        COLUNAS_ESTOQUES_FINAL
    ]
    .copy()
)


# ------------------------------------------------------------
# Converter incerteza relativa para percentual
# mantendo explicitamente a unidade
# ------------------------------------------------------------

COLUNAS_INCERTEZA_RELATIVA = [
    "ctotal_t0_incerteza",
    "ctotal_t1_incerteza",
    "soc_t0_incerteza",
    "soc_t1_incerteza",
    "cveg_t0_incerteza",
    "cveg_t1_incerteza"
]


for coluna in COLUNAS_INCERTEZA_RELATIVA:

    nova_coluna = (
        coluna
        .replace(
            "_incerteza",
            "_incerteza_pct"
        )
    )

    brluc_estoques_final[
        nova_coluna
    ] = (
        brluc_estoques_final[
            coluna
        ]
        * 100
    )


# Remover a versão proporcional para evitar ambiguidade
brluc_estoques_final = (
    brluc_estoques_final
    .drop(
        columns=COLUNAS_INCERTEZA_RELATIVA
    )
)


print(
    "Dimensão final do Appendix B:"
)

print(
    brluc_estoques_final.shape
)


print(
    "\nDuplicatas:"
)

print(
    brluc_estoques_final
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


display(
    brluc_estoques_final.head()
)

Dimensão final do Appendix B:
(1658, 36)

Duplicatas:
0


,codigo_ibge,classe_brluc_t0,classe_brluc_t1,ctotal_t0_t_c_ha,ctotal_t0_se,ctotal_t0_ic95_inf,ctotal_t0_ic95_sup,ctotal_t1_t_c_ha,ctotal_t1_se,ctotal_t1_ic95_inf,...,cveg_t1_ic95_sup,delta_soc_classes_brluc_t1_t0_t_c_ha,delta_cveg_classes_brluc_t1_t0_t_c_ha,delta_ctotal_classes_brluc_t1_t0_t_c_ha,ctotal_t0_incerteza_pct,ctotal_t1_incerteza_pct,soc_t0_incerteza_pct,soc_t1_incerteza_pct,cveg_t0_incerteza_pct,cveg_t1_incerteza_pct
0,4100103,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",31.986914,5.996241,21.543618,44.477164,32.919802,6.205433,21.803863,...,8.266892,0.932888,0.0,0.932888,39.047998,38.394197,42.136504,43.265303,75.671727,75.671727
1,4100202,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",42.606842,5.959154,31.404718,54.445569,44.790669,6.310514,33.105301,...,8.266892,2.183826,0.0,2.183826,27.785975,28.595182,30.596729,30.309872,75.671727,75.671727
2,4100301,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",44.657490,5.831785,33.545015,55.916646,45.580334,5.959076,34.564422,...,8.266892,0.922843,0.0,0.922843,25.212243,25.982561,27.557669,27.390285,75.671727,75.671727
3,4100400,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",75.820765,7.609789,61.054837,90.463825,78.467974,7.921938,63.623031,...,8.266892,2.647209,0.0,2.647209,19.474781,20.076554,20.392620,20.796547,75.671727,75.671727
4,4100459,"Cropland, temporary, generic, average tillage ...","Cropland, temporary, generic, average tillage ...",58.723620,6.547916,46.533862,71.900557,59.960428,6.729316,47.471231,...,8.266892,1.236808,0.0,1.236808,22.438905,22.677040,23.556670,23.323328,75.671727,75.671727


In [56]:
# ============================================================
# BRLUC CONSOLIDADO — APPENDIX C + APPENDIX B
# ============================================================

brluc_consolidado = (
    brluc_curated
    .merge(
        brluc_estoques_final,
        on="codigo_ibge",
        how="left",
        validate="one_to_one",
        indicator=True
    )
)


print("=" * 70)
print("BRLUC CONSOLIDADO")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    brluc_consolidado.shape
)


print(
    "\nStatus do merge:"
)

display(
    brluc_consolidado[
        "_merge"
    ]
    .value_counts()
)


print(
    "\nMunicípios distintos:"
)

print(
    brluc_consolidado[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    brluc_consolidado
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


# ------------------------------------------------------------
# Remover controle de merge após validação
# ------------------------------------------------------------

brluc_consolidado = (
    brluc_consolidado
    .drop(
        columns=[
            "_merge"
        ]
    )
)


print(
    "\nDimensão após remover _merge:"
)

print(
    brluc_consolidado.shape
)


display(
    brluc_consolidado.head()
)

BRLUC CONSOLIDADO

Dimensão:
(1658, 69)

Status do merge:


_merge
both          1658
left_only        0
right_only       0
Name: count, dtype: int64


Municípios distintos:
1658

Duplicatas:
0

Dimensão após remover _merge:
(1658, 68)


,codigo_ibge,municipio,estado,uf,regiao,cultura,periodo_inicio_brluc,periodo_fim_brluc,area_origem_temporaria_ha,area_origem_soja_ha,...,cveg_t1_ic95_sup,delta_soc_classes_brluc_t1_t0_t_c_ha,delta_cveg_classes_brluc_t1_t0_t_c_ha,delta_ctotal_classes_brluc_t1_t0_t_c_ha,ctotal_t0_incerteza_pct,ctotal_t1_incerteza_pct,soc_t0_incerteza_pct,soc_t1_incerteza_pct,cveg_t0_incerteza_pct,cveg_t1_incerteza_pct
0,4100103,Abatiá,Paraná,PR,Sul,Soja,2000,2019,2488.168493,371.845266,...,8.266892,0.932888,0.0,0.932888,39.047998,38.394197,42.136504,43.265303,75.671727,75.671727
1,4100202,Adrianópolis,Paraná,PR,Sul,Soja,2000,2019,1.450002,1.302738,...,8.266892,2.183826,0.0,2.183826,27.785975,28.595182,30.596729,30.309872,75.671727,75.671727
2,4100301,Agudos do Sul,Paraná,PR,Sul,Soja,2000,2019,326.200765,1.287731,...,8.266892,0.922843,0.0,0.922843,25.212243,25.982561,27.557669,27.390285,75.671727,75.671727
3,4100400,Almirante Tamandaré,Paraná,PR,Sul,Soja,2000,2019,508.821336,17.580535,...,8.266892,2.647209,0.0,2.647209,19.474781,20.076554,20.392620,20.796547,75.671727,75.671727
4,4100459,Altamira do Paraná,Paraná,PR,Sul,Soja,2000,2019,138.539817,0.000000,...,8.266892,1.236808,0.0,1.236808,22.438905,22.677040,23.556670,23.323328,75.671727,75.671727


In [57]:
# ============================================================
# APPENDIX B — INVESTIGAÇÃO DA FÓRMULA DE UNC
# ============================================================

testes_incerteza_completa = [
    (
        "ctotal_t0",
        "ctotal_t0_t_c_ha",
        "ctotal_t0_ic95_inf",
        "ctotal_t0_ic95_sup",
        "ctotal_t0_incerteza"
    ),
    (
        "ctotal_t1",
        "ctotal_t1_t_c_ha",
        "ctotal_t1_ic95_inf",
        "ctotal_t1_ic95_sup",
        "ctotal_t1_incerteza"
    ),
    (
        "soc_t0",
        "soc_t0_t_c_ha",
        "soc_t0_ic95_inf",
        "soc_t0_ic95_sup",
        "soc_t0_incerteza"
    ),
    (
        "soc_t1",
        "soc_t1_t_c_ha",
        "soc_t1_ic95_inf",
        "soc_t1_ic95_sup",
        "soc_t1_incerteza"
    ),
    (
        "cveg_t0",
        "cveg_t0_t_c_ha",
        "cveg_t0_ic95_inf",
        "cveg_t0_ic95_sup",
        "cveg_t0_incerteza"
    ),
    (
        "cveg_t1",
        "cveg_t1_t_c_ha",
        "cveg_t1_ic95_inf",
        "cveg_t1_ic95_sup",
        "cveg_t1_incerteza"
    )
]


for nome, estimativa, ic_inf, ic_sup, incerteza in testes_incerteza_completa:

    base_estimativa = (
        estoques_soja_projeto_b[
            estimativa
        ]
        .replace(0, np.nan)
    )


    lado_inferior = (
        (
            estoques_soja_projeto_b[
                estimativa
            ]
            -
            estoques_soja_projeto_b[
                ic_inf
            ]
        )
        /
        base_estimativa
    )


    lado_superior = (
        (
            estoques_soja_projeto_b[
                ic_sup
            ]
            -
            estoques_soja_projeto_b[
                estimativa
            ]
        )
        /
        base_estimativa
    )


    maior_lado = np.maximum(
        lado_inferior,
        lado_superior
    )


    diferenca = (
        maior_lado
        -
        estoques_soja_projeto_b[
            incerteza
        ]
    ).abs()


    print("\n" + "=" * 70)
    print(nome)

    print(
        "Maior diferença:",
        diferenca.max()
    )

    print(
        "Falhas > 0.000001:",
        (
            diferenca
            > 0.000001
        )
        .sum()
    )


ctotal_t0
Maior diferença: 6.411537967210279e-15
Falhas > 0.000001: 0

ctotal_t1
Maior diferença: 6.106226635438361e-15
Falhas > 0.000001: 0

soc_t0
Maior diferença: 5.745404152435185e-15
Falhas > 0.000001: 0

soc_t1
Maior diferença: 5.800915303666443e-15
Falhas > 0.000001: 0

cveg_t0
Maior diferença: 1.4432899320127035e-15
Falhas > 0.000001: 0

cveg_t1
Maior diferença: 1.4432899320127035e-15
Falhas > 0.000001: 0


In [58]:
# ============================================================
# APPENDIX B — ASSIMETRIA DOS INTERVALOS
# ============================================================

for nome, estimativa, ic_inf, ic_sup, incerteza in testes_incerteza_completa:

    base_estimativa = (
        estoques_soja_projeto_b[
            estimativa
        ]
        .replace(0, np.nan)
    )


    lado_inferior = (
        (
            estoques_soja_projeto_b[
                estimativa
            ]
            -
            estoques_soja_projeto_b[
                ic_inf
            ]
        )
        /
        base_estimativa
    )


    lado_superior = (
        (
            estoques_soja_projeto_b[
                ic_sup
            ]
            -
            estoques_soja_projeto_b[
                estimativa
            ]
        )
        /
        base_estimativa
    )


    print("\n" + "=" * 70)
    print(nome)

    print(
        "IC inferior é o lado maior:",
        (
            lado_inferior
            >
            lado_superior
        )
        .sum()
    )

    print(
        "IC superior é o lado maior:",
        (
            lado_superior
            >
            lado_inferior
        )
        .sum()
    )

    print(
        "Lados praticamente iguais:",
        np.isclose(
            lado_inferior,
            lado_superior
        )
        .sum()
    )


ctotal_t0
IC inferior é o lado maior: 182
IC superior é o lado maior: 1476
Lados praticamente iguais: 0

ctotal_t1
IC inferior é o lado maior: 226
IC superior é o lado maior: 1432
Lados praticamente iguais: 0

soc_t0
IC inferior é o lado maior: 204
IC superior é o lado maior: 1454
Lados praticamente iguais: 0

soc_t1
IC inferior é o lado maior: 182
IC superior é o lado maior: 1476
Lados praticamente iguais: 0

cveg_t0
IC inferior é o lado maior: 0
IC superior é o lado maior: 1658
Lados praticamente iguais: 0

cveg_t1
IC inferior é o lado maior: 0
IC superior é o lado maior: 1658
Lados praticamente iguais: 0


In [59]:
# ============================================================
# APPENDIX B — CAMADA FINAL CORRIGIDA
# Preserva UNC original + versão percentual
# ============================================================

COLUNAS_ESTOQUES_FINAL = [
    "codigo_ibge",

    # --------------------------------------------------------
    # Classes metodológicas associadas à soja
    # --------------------------------------------------------
    "classe_brluc_t0",
    "classe_brluc_t1",

    # --------------------------------------------------------
    # CTOTAL
    # --------------------------------------------------------
    "ctotal_t0_t_c_ha",
    "ctotal_t0_se",
    "ctotal_t0_ic95_inf",
    "ctotal_t0_ic95_sup",
    "ctotal_t0_incerteza",

    "ctotal_t1_t_c_ha",
    "ctotal_t1_se",
    "ctotal_t1_ic95_inf",
    "ctotal_t1_ic95_sup",
    "ctotal_t1_incerteza",

    # --------------------------------------------------------
    # SOC
    # --------------------------------------------------------
    "soc_t0_t_c_ha",
    "soc_t0_se",
    "soc_t0_ic95_inf",
    "soc_t0_ic95_sup",
    "soc_t0_incerteza",

    "soc_t1_t_c_ha",
    "soc_t1_se",
    "soc_t1_ic95_inf",
    "soc_t1_ic95_sup",
    "soc_t1_incerteza",

    # --------------------------------------------------------
    # CVEG
    # --------------------------------------------------------
    "cveg_t0_t_c_ha",
    "cveg_t0_se",
    "cveg_t0_ic95_inf",
    "cveg_t0_ic95_sup",
    "cveg_t0_incerteza",

    "cveg_t1_t_c_ha",
    "cveg_t1_se",
    "cveg_t1_ic95_inf",
    "cveg_t1_ic95_sup",
    "cveg_t1_incerteza",

    # --------------------------------------------------------
    # Deltas entre as classes BRLUC
    # --------------------------------------------------------
    "delta_soc_classes_brluc_t1_t0_t_c_ha",
    "delta_cveg_classes_brluc_t1_t0_t_c_ha",
    "delta_ctotal_classes_brluc_t1_t0_t_c_ha"
]


brluc_estoques_final = (
    brluc_estoques_curated[
        COLUNAS_ESTOQUES_FINAL
    ]
    .copy()
)


# ------------------------------------------------------------
# Renomear UNC original para deixar claro que é fração relativa
# ------------------------------------------------------------

MAPA_INCERTEZA_RELATIVA = {
    "ctotal_t0_incerteza":
        "ctotal_t0_incerteza_relativa",

    "ctotal_t1_incerteza":
        "ctotal_t1_incerteza_relativa",

    "soc_t0_incerteza":
        "soc_t0_incerteza_relativa",

    "soc_t1_incerteza":
        "soc_t1_incerteza_relativa",

    "cveg_t0_incerteza":
        "cveg_t0_incerteza_relativa",

    "cveg_t1_incerteza":
        "cveg_t1_incerteza_relativa"
}


brluc_estoques_final = (
    brluc_estoques_final
    .rename(
        columns=MAPA_INCERTEZA_RELATIVA
    )
)


# ------------------------------------------------------------
# Criar versão em percentual
# ------------------------------------------------------------

for coluna in MAPA_INCERTEZA_RELATIVA.values():

    coluna_pct = (
        coluna
        .replace(
            "_relativa",
            "_pct"
        )
    )

    brluc_estoques_final[
        coluna_pct
    ] = (
        brluc_estoques_final[
            coluna
        ]
        * 100
    )


print("=" * 70)
print("APPENDIX B — CAMADA FINAL CORRIGIDA")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    brluc_estoques_final.shape
)


print(
    "\nMunicípios:"
)

print(
    brluc_estoques_final[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    brluc_estoques_final
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


display(
    brluc_estoques_final[
        [
            "codigo_ibge",

            "ctotal_t0_incerteza_relativa",
            "ctotal_t0_incerteza_pct",

            "soc_t0_incerteza_relativa",
            "soc_t0_incerteza_pct",

            "cveg_t0_incerteza_relativa",
            "cveg_t0_incerteza_pct"
        ]
    ]
    .head()
)

APPENDIX B — CAMADA FINAL CORRIGIDA

Dimensão:
(1658, 42)

Municípios:
1658

Duplicatas:
0


,codigo_ibge,ctotal_t0_incerteza_relativa,ctotal_t0_incerteza_pct,soc_t0_incerteza_relativa,soc_t0_incerteza_pct,cveg_t0_incerteza_relativa,cveg_t0_incerteza_pct
0,4100103,0.390480,39.047998,0.421365,42.136504,0.756717,75.671727
1,4100202,0.277860,27.785975,0.305967,30.596729,0.756717,75.671727
2,4100301,0.252122,25.212243,0.275577,27.557669,0.756717,75.671727
3,4100400,0.194748,19.474781,0.203926,20.392620,0.756717,75.671727
4,4100459,0.224389,22.438905,0.235567,23.556670,0.756717,75.671727


In [60]:
# ============================================================
# BRLUC CONSOLIDADO — APPENDIX C + APPENDIX B CORRIGIDO
# ============================================================

brluc_consolidado = (
    brluc_curated
    .merge(
        brluc_estoques_final,
        on="codigo_ibge",
        how="left",
        validate="one_to_one",
        indicator=True
    )
)


print("=" * 70)
print("BRLUC CONSOLIDADO — VERSÃO CORRIGIDA")
print("=" * 70)


print(
    "\nDimensão antes de remover _merge:"
)

print(
    brluc_consolidado.shape
)


print(
    "\nStatus do merge:"
)

display(
    brluc_consolidado[
        "_merge"
    ]
    .value_counts()
)


print(
    "\nMunicípios distintos:"
)

print(
    brluc_consolidado[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    brluc_consolidado
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


brluc_consolidado = (
    brluc_consolidado
    .drop(
        columns="_merge"
    )
)


print(
    "\nDimensão final:"
)

print(
    brluc_consolidado.shape
)

BRLUC CONSOLIDADO — VERSÃO CORRIGIDA

Dimensão antes de remover _merge:
(1658, 75)

Status do merge:


_merge
both          1658
left_only        0
right_only       0
Name: count, dtype: int64


Municípios distintos:
1658

Duplicatas:
0

Dimensão final:
(1658, 74)


In [61]:
# ============================================================
# VALIDAÇÃO FINAL — BRLUC CONSOLIDADO
# ============================================================

print("=" * 70)
print("VALIDAÇÃO FINAL — BRLUC CONSOLIDADO")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    brluc_consolidado.shape
)


print(
    "\nMunicípios distintos:"
)

print(
    brluc_consolidado[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    brluc_consolidado
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


# ------------------------------------------------------------
# Appendix C
# ------------------------------------------------------------

print(
    "\nEmissões absolutas negativas:"
)

print(
    (
        brluc_consolidado[
            "emissao_absoluta_co2_t_ano"
        ]
        < 0
    )
    .sum()
)


print(
    "\nTaxas de emissão negativas:"
)

print(
    (
        brluc_consolidado[
            "taxa_emissao_co2_t_ha_ano"
        ]
        < 0
    )
    .sum()
)


print(
    "\nPercentuais de conversão NULL:"
)

print(
    brluc_consolidado[
        "percentual_conversao_para_soja_pct"
    ]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Appendix B
# ------------------------------------------------------------

print(
    "\nSOC t0 nulo:"
)

print(
    brluc_consolidado[
        "soc_t0_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nSOC t1 nulo:"
)

print(
    brluc_consolidado[
        "soc_t1_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nCtotal t0 nulo:"
)

print(
    brluc_consolidado[
        "ctotal_t0_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nCveg t0 nulo:"
)

print(
    brluc_consolidado[
        "cveg_t0_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nDelta SOC:"
)

print(
    "Positivo:",
    (
        brluc_consolidado[
            "delta_soc_classes_brluc_t1_t0_t_c_ha"
        ]
        > 0
    ).sum()
)

print(
    "Zero:",
    np.isclose(
        brluc_consolidado[
            "delta_soc_classes_brluc_t1_t0_t_c_ha"
        ],
        0
    ).sum()
)

print(
    "Negativo:",
    (
        brluc_consolidado[
            "delta_soc_classes_brluc_t1_t0_t_c_ha"
        ]
        < 0
    ).sum()
)


# ------------------------------------------------------------
# Ctotal = SOC + Cveg
# ------------------------------------------------------------

dif_ctotal_t0_final = (
    brluc_consolidado[
        "ctotal_t0_t_c_ha"
    ]
    -
    (
        brluc_consolidado[
            "soc_t0_t_c_ha"
        ]
        +
        brluc_consolidado[
            "cveg_t0_t_c_ha"
        ]
    )
).abs()


dif_ctotal_t1_final = (
    brluc_consolidado[
        "ctotal_t1_t_c_ha"
    ]
    -
    (
        brluc_consolidado[
            "soc_t1_t_c_ha"
        ]
        +
        brluc_consolidado[
            "cveg_t1_t_c_ha"
        ]
    )
).abs()


print(
    "\nFalhas Ctotal = SOC + Cveg | t0:"
)

print(
    (
        dif_ctotal_t0_final
        > 0.000001
    )
    .sum()
)


print(
    "\nFalhas Ctotal = SOC + Cveg | t1:"
)

print(
    (
        dif_ctotal_t1_final
        > 0.000001
    )
    .sum()
)

VALIDAÇÃO FINAL — BRLUC CONSOLIDADO

Dimensão:
(1658, 74)

Municípios distintos:
1658

Duplicatas:
0

Emissões absolutas negativas:
17

Taxas de emissão negativas:
17

Percentuais de conversão NULL:
102

SOC t0 nulo:
0

SOC t1 nulo:
0

Ctotal t0 nulo:
0

Cveg t0 nulo:
0

Delta SOC:
Positivo: 1482
Zero: 20
Negativo: 156

Falhas Ctotal = SOC + Cveg | t0:
0

Falhas Ctotal = SOC + Cveg | t1:
0


### Interpretação da incerteza no Appendix B

A coluna `Unc` do Appendix B representa a maior distância relativa
entre a estimativa central e os limites inferior ou superior do
intervalo de confiança de 95%.

A relação foi validada para Ctotal, SOC e Cveg, tanto para as classes
associadas a t0 quanto a t1:

`Unc = max((estimativa - IC95_inf) / estimativa,
           (IC95_sup - estimativa) / estimativa)`

Embora a documentação apresente a variável como `Uncertainty (%)`,
os valores no workbook são armazenados como fração relativa.
Por isso, a camada Curated preserva o valor relativo original e cria
também uma versão multiplicada por 100, explicitamente identificada
com o sufixo `_pct`.

In [62]:
# ============================================================
# EXPORTAÇÃO FINAL — APPENDIX B + BRLUC CONSOLIDADO
# ============================================================

BRLUC_CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ARQUIVO_BRLUC_ESTOQUES = (
    BRLUC_CURATED_DIR
    / "brluc_estoques_carbono_soja_centro_oeste_sul_2000_2019.csv"
)


ARQUIVO_BRLUC_CONSOLIDADO = (
    BRLUC_CURATED_DIR
    / "brluc_soja_consolidado_centro_oeste_sul_2000_2019.csv"
)


# ------------------------------------------------------------
# Preparar Appendix B
# ------------------------------------------------------------

brluc_estoques_exportacao = (
    brluc_estoques_final
    .copy()
)


brluc_estoques_exportacao[
    "codigo_ibge"
] = (
    brluc_estoques_exportacao[
        "codigo_ibge"
    ]
    .astype("string")
    .str.zfill(7)
)


brluc_estoques_exportacao = (
    brluc_estoques_exportacao
    .sort_values(
        by="codigo_ibge"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Preparar consolidado
# ------------------------------------------------------------

brluc_consolidado_exportacao = (
    brluc_consolidado
    .copy()
)


brluc_consolidado_exportacao[
    "codigo_ibge"
] = (
    brluc_consolidado_exportacao[
        "codigo_ibge"
    ]
    .astype("string")
    .str.zfill(7)
)


brluc_consolidado_exportacao = (
    brluc_consolidado_exportacao
    .sort_values(
        by=[
            "uf",
            "codigo_ibge"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Exportar
# ------------------------------------------------------------

brluc_estoques_exportacao.to_csv(
    ARQUIVO_BRLUC_ESTOQUES,
    index=False,
    encoding="utf-8-sig"
)


brluc_consolidado_exportacao.to_csv(
    ARQUIVO_BRLUC_CONSOLIDADO,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 70)
print("EXPORTAÇÃO FINAL — BRLUC")
print("=" * 70)


print(
    "\nAppendix B — estoques:"
)

print(
    ARQUIVO_BRLUC_ESTOQUES
)

print(
    "Existe:",
    ARQUIVO_BRLUC_ESTOQUES.exists()
)

print(
    "Dimensão:",
    brluc_estoques_exportacao.shape
)

print(
    "Tamanho:",
    f"{ARQUIVO_BRLUC_ESTOQUES.stat().st_size / (1024 * 1024):.2f} MB"
)


print(
    "\nBRLUC consolidado:"
)

print(
    ARQUIVO_BRLUC_CONSOLIDADO
)

print(
    "Existe:",
    ARQUIVO_BRLUC_CONSOLIDADO.exists()
)

print(
    "Dimensão:",
    brluc_consolidado_exportacao.shape
)

print(
    "Tamanho:",
    f"{ARQUIVO_BRLUC_CONSOLIDADO.stat().st_size / (1024 * 1024):.2f} MB"
)

EXPORTAÇÃO FINAL — BRLUC

Appendix B — estoques:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\embrapa_brluc\municipio\brluc_estoques_carbono_soja_centro_oeste_sul_2000_2019.csv
Existe: True
Dimensão: (1658, 42)
Tamanho: 1.25 MB

BRLUC consolidado:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\embrapa_brluc\municipio\brluc_soja_consolidado_centro_oeste_sul_2000_2019.csv
Existe: True
Dimensão: (1658, 74)
Tamanho: 1.88 MB


In [63]:
# ============================================================
# VALIDAÇÃO PÓS-EXPORTAÇÃO — BRLUC CONSOLIDADO
# ============================================================

brluc_consolidado_recarregado = pd.read_csv(
    ARQUIVO_BRLUC_CONSOLIDADO,
    dtype={
        "codigo_ibge": "string"
    },
    encoding="utf-8-sig"
)


brluc_consolidado_recarregado[
    "codigo_ibge"
] = (
    brluc_consolidado_recarregado[
        "codigo_ibge"
    ]
    .str.zfill(7)
)


print("=" * 70)
print("VALIDAÇÃO DO CSV CONSOLIDADO")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    brluc_consolidado_recarregado.shape
)


print(
    "\nMunicípios distintos:"
)

print(
    brluc_consolidado_recarregado[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas:"
)

print(
    brluc_consolidado_recarregado
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nMesmo universo do DataFrame original:"
)

print(
    set(
        brluc_consolidado_recarregado[
            "codigo_ibge"
        ]
    )
    ==
    set(
        brluc_consolidado_exportacao[
            "codigo_ibge"
        ]
    )
)


print(
    "\nPercentual conversão NULL:"
)

print(
    brluc_consolidado_recarregado[
        "percentual_conversao_para_soja_pct"
    ]
    .isna()
    .sum()
)


print(
    "\nIC95 emissão absoluta inferior NULL:"
)

print(
    brluc_consolidado_recarregado[
        "emissao_absoluta_co2_ic95_inf"
    ]
    .isna()
    .sum()
)


print(
    "\nIC95 taxa emissão inferior NULL:"
)

print(
    brluc_consolidado_recarregado[
        "taxa_emissao_co2_ic95_inf"
    ]
    .isna()
    .sum()
)


print(
    "\nSOC t0 NULL:"
)

print(
    brluc_consolidado_recarregado[
        "soc_t0_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nSOC t1 NULL:"
)

print(
    brluc_consolidado_recarregado[
        "soc_t1_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nEmissões negativas:"
)

print(
    (
        brluc_consolidado_recarregado[
            "emissao_absoluta_co2_t_ano"
        ]
        < 0
    )
    .sum()
)


print(
    "\nTaxas negativas:"
)

print(
    (
        brluc_consolidado_recarregado[
            "taxa_emissao_co2_t_ha_ano"
        ]
        < 0
    )
    .sum()
)

VALIDAÇÃO DO CSV CONSOLIDADO

Dimensão:
(1658, 74)

Municípios distintos:
1658

Duplicatas:
0

Mesmo universo do DataFrame original:
True

Percentual conversão NULL:
102

IC95 emissão absoluta inferior NULL:
102

IC95 taxa emissão inferior NULL:
118

SOC t0 NULL:
0

SOC t1 NULL:
0

Emissões negativas:
17

Taxas negativas:
17


In [64]:
# ============================================================
# CONTROLES MATEMÁTICOS — CSV BRLUC CONSOLIDADO
# ============================================================

dif_ctotal_t0_csv = (
    brluc_consolidado_recarregado[
        "ctotal_t0_t_c_ha"
    ]
    -
    (
        brluc_consolidado_recarregado[
            "soc_t0_t_c_ha"
        ]
        +
        brluc_consolidado_recarregado[
            "cveg_t0_t_c_ha"
        ]
    )
).abs()


dif_ctotal_t1_csv = (
    brluc_consolidado_recarregado[
        "ctotal_t1_t_c_ha"
    ]
    -
    (
        brluc_consolidado_recarregado[
            "soc_t1_t_c_ha"
        ]
        +
        brluc_consolidado_recarregado[
            "cveg_t1_t_c_ha"
        ]
    )
).abs()


dif_delta_soc_ctotal_csv = (
    brluc_consolidado_recarregado[
        "delta_soc_classes_brluc_t1_t0_t_c_ha"
    ]
    -
    brluc_consolidado_recarregado[
        "delta_ctotal_classes_brluc_t1_t0_t_c_ha"
    ]
).abs()


print("=" * 70)
print("CONTROLES MATEMÁTICOS — CSV FINAL")
print("=" * 70)


print(
    "\nMaior diferença Ctotal = SOC + Cveg | t0:"
)

print(
    dif_ctotal_t0_csv.max()
)


print(
    "Falhas > 0.000001:",
    (
        dif_ctotal_t0_csv
        > 0.000001
    ).sum()
)


print(
    "\nMaior diferença Ctotal = SOC + Cveg | t1:"
)

print(
    dif_ctotal_t1_csv.max()
)


print(
    "Falhas > 0.000001:",
    (
        dif_ctotal_t1_csv
        > 0.000001
    ).sum()
)


print(
    "\nMaior diferença Delta SOC × Delta Ctotal:"
)

print(
    dif_delta_soc_ctotal_csv.max()
)


print(
    "Falhas > 0.000001:",
    (
        dif_delta_soc_ctotal_csv
        > 0.000001
    ).sum()
)

CONTROLES MATEMÁTICOS — CSV FINAL

Maior diferença Ctotal = SOC + Cveg | t0:
8.526512829121202e-14
Falhas > 0.000001: 0

Maior diferença Ctotal = SOC + Cveg | t1:
8.526512829121202e-14
Falhas > 0.000001: 0

Maior diferença Delta SOC × Delta Ctotal:
1.141309269314661e-13
Falhas > 0.000001: 0


## Conclusão do processamento Embrapa BRLUC

O processamento do BRLUC 2.1 foi realizado diretamente a partir dos
arquivos originais dos Appendices B e C.

Para o escopo do projeto foram selecionados os municípios das regiões
Centro-Oeste e Sul e a categoria associada à soja.

### Appendix C

Foram incorporados indicadores de mudança de uso da terra, incluindo:

- áreas de origem das diferentes classes;
- permanência soja → soja;
- conversão de outras classes → soja;
- percentuais de persistência e conversão;
- emissão absoluta de CO₂;
- taxa de emissão de CO₂;
- erros-padrão e intervalos de confiança.

Os resultados representam a transição BRLUC entre t0 = 2000 e
t1 = 2019.

### Appendix B

Foram identificadas as classes de estoque utilizadas pelo BRLUC para
representar a soja:

- t0:
  `Cropland, temporary, generic, average tillage 2006, medium`

- t1:
  `Cropland, temporary, generic, average tillage 2017, medium`

Para essas classes foram extraídos:

- SOC — estoque de carbono orgânico do solo;
- Cveg — estoque de carbono da vegetação;
- Ctotal — estoque total de carbono;
- erros-padrão;
- intervalos de confiança de 95%;
- incerteza relativa;
- incerteza em percentual.

Foi confirmado matematicamente que:

`Ctotal = SOC + Cveg`

para todos os 1.658 municípios do recorte.

A diferença entre as classes t1 e t0 foi mantida como indicador
metodológico BRLUC e não deve ser interpretada diretamente como
sequestro realizado, crédito de carbono gerado ou quantidade
comercializável de créditos.

### Resultado final

A camada consolidada BRLUC contém:

- 1.658 municípios;
- 74 variáveis;
- nenhuma duplicidade de código IBGE.

O BRLUC será tratado nas integrações posteriores como uma camada
municipal estrutural referente à transição 2000–2019.

Não será criada artificialmente uma série anual do BRLUC para
2019–2024.